# Diabetes Glycemic Control Prediction: Full Pipeline

**Team:** Me, Myself, and AI (Gabriel Kettering, Susie Kim, Rishab Haldar, Anthony Kirchner)

**Objective:** Predict whether a patient with diabetes will have an uncontrolled A1c (>8%) in 2025, using 12 months of prior EHR data.

**Pipeline phases:**
1. Setup and data loading
2. Descriptive statistics (clinical binning by outcome)
3. Data cleaning
4. Feature engineering (97 features down to 25)
5. Model comparison (6 models x 5-fold stratified CV)
6. Permutation importance and feature reduction
7. Final CatBoost model with shared train/predict functions
8. Honest vs hardcoded AUC comparison (the labeling discovery)

**Required files in Google Drive root:**
- `DM_Features.csv` (62,425 patients, ~40 raw columns)
- `DM_Control_2025.csv` (binary outcome)
- `TEST_SET_DM_Features.csv` (15,607 held-out test patients)

In [ ]:
# ── CELL 1: Setup ──
!pip install -q catboost

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import sys

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    roc_curve, precision_recall_curve, confusion_matrix,
    accuracy_score, classification_report,
)

import lightgbm as lgb
import xgboost as xgb
import catboost as cb

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

drive_path = "/content/drive/My Drive/"
print("Setup complete.")

In [ ]:
# ── CELL 2: Load raw data ──
features_raw = pd.read_csv(drive_path + "DM_Features.csv", index_col=0)
control_raw = pd.read_csv(drive_path + "DM_Control_2025.csv", index_col=0)
df_full = features_raw.join(control_raw)
outcome = "a1c 2025 Uncontrolled"

print(f"Full dataset: {len(df_full):,} patients x {len(df_full.columns)} columns")
print(f"Outcome distribution:")
print(f"  Controlled:   {(~df_full[outcome]).sum():,} ({(~df_full[outcome]).mean()*100:.1f}%)")
print(f"  Uncontrolled: {df_full[outcome].sum():,} ({df_full[outcome].mean()*100:.1f}%)")

# Identify the labeling pattern
ref_col = "a1c 2025-collection date-time-days from reference"
null_2025 = df_full[ref_col].isna()
print(f"\nNull 2025 date: {null_2025.sum():,} ({null_2025.mean()*100:.1f}%)")
print(f"All null-date patients labeled controlled: {(df_full.loc[null_2025, outcome] == False).all()}")
print(f"Any non-null-date patients labeled controlled: {(df_full.loc[~null_2025, outcome] == False).any()}")

## Phase 1: Descriptive Statistics and EDA Figures

Comprehensive descriptive analysis on the raw dataset. Text-based clinical binning (all 62,425 patients stratified by outcome), then visual EDA figures on the 35,808-patient subset with non-null 2025 dates (before A1c outlier filtering). Includes missingness profiling, demographics, A1c distributions at each visit, threshold analysis, clinical measures, A1c trajectory plots stratified by demographics/comorbidities/medications/ADI, correlation matrix, preliminary feature importance, and partial dependence plots.

In [ ]:
# ── CELL 3: Comprehensive descriptive statistics ──
# Every variable binned using clinical thresholds, stratified by outcome
outcome_col = outcome

def print_binned_split(series, label):
    print(f"\n{'='*70}")
    print(f"  {label}")
    print(f"  Total Missing: {series.isna().sum():,}")
    print("-"*70)

    combined = pd.DataFrame({"value": series, "outcome": df_full[outcome_col]})

    controlled = combined[combined["outcome"] == False]["value"]
    uncontrolled = combined[combined["outcome"] == True]["value"]

    ctrl_counts = controlled.value_counts(dropna=False, sort=False)
    unctrl_counts = uncontrolled.value_counts(dropna=False, sort=False)

    summary = pd.DataFrame({
        f"Controlled (n={len(controlled):,})": ctrl_counts,
        f"Uncontrolled (n={len(uncontrolled):,})": unctrl_counts
    }).fillna(0).astype(int)

    summary["Ctrl %"] = (summary.iloc[:, 0] / len(controlled) * 100).round(1)
    summary["Unctrl %"] = (summary.iloc[:, 1] / len(uncontrolled) * 100).round(1)

    summary = summary[[summary.columns[0], "Ctrl %", summary.columns[1], "Unctrl %"]]
    print(summary.to_string())

# ──────────────────────────────────────────────────────────────
# OUTCOME DISTRIBUTION
# ──────────────────────────────────────────────────────────────
print("="*70)
print("  OUTCOME DISTRIBUTION")
print("-"*70)
print(df_full[outcome].value_counts().to_string())

# ──────────────────────────────────────────────────────────────
# 1. DEMOGRAPHICS
# ──────────────────────────────────────────────────────────────
current_year = 2025
df_full["age"] = current_year - df_full["date of birth"]
age_bins = [0, 29, 39, 49, 59, 69, 79, 200]
age_labels = ["18-29", "30-39", "40-49", "50-59", "60-69", "70-79", "80+"]
df_full["age_group"] = pd.cut(df_full["age"], bins=age_bins, labels=age_labels, right=True)
print_binned_split(df_full["age_group"], "Age Group")

for col in ["gender at birth", "ethnicity", "race - primary"]:
    print_binned_split(df_full[col], col)

# ──────────────────────────────────────────────────────────────
# 2. A1C VALUES (ADA thresholds)
# ──────────────────────────────────────────────────────────────
a1c_bins = [0, 5.7, 6.5, 7.0, 8.0, 9.0, 100]
a1c_labels = ["<5.7 Normal", "5.7-6.4 Prediabetes", "6.5-7.0 Controlled",
              "7.1-8.0 Suboptimal", "8.1-9.0 Poor", ">9.0 Very Poor"]

for col in [c for c in df_full.columns if "estimated result" in c and "a1c" in c]:
    binned = pd.cut(df_full[col], bins=a1c_bins, labels=a1c_labels, right=True)
    print_binned_split(binned, col)

# ──────────────────────────────────────────────────────────────
# 3. BMI (WHO classification)
# ──────────────────────────────────────────────────────────────
df_full["bmi_raw"] = df_full["weight-estimated result"] / ((df_full["height-estimated result"] / 100) ** 2)
bmi_bins = [0, 18.5, 25, 30, 35, 40, 200]
bmi_labels = ["<18.5 Underweight", "18.5-24.9 Normal", "25-29.9 Overweight",
              "30-34.9 Obesity I", "35-39.9 Obesity II", ">=40 Obesity III"]
df_full["bmi_group"] = pd.cut(df_full["bmi_raw"], bins=bmi_bins, labels=bmi_labels, right=False)
print_binned_split(df_full["bmi_group"], "BMI (WHO Classification)")

# ──────────────────────────────────────────────────────────────
# 4. LIPIDS
# ──────────────────────────────────────────────────────────────
ldl_bins = [0, 70, 100, 130, 160, 9999]
ldl_labels = ["<70 Optimal", "70-99 Near Optimal", "100-129 Borderline",
              "130-159 High", ">=160 Very High"]
df_full["ldl_group"] = pd.cut(df_full["ldl-estimated result"], bins=ldl_bins, labels=ldl_labels, right=False)
print_binned_split(df_full["ldl_group"], "LDL (mg/dL)")

hdl_bins = [0, 40, 60, 9999]
hdl_labels = ["<40 Low (Risk)", "40-59 Normal", ">=60 Protective"]
df_full["hdl_group"] = pd.cut(df_full["hdl-estimated result"], bins=hdl_bins, labels=hdl_labels, right=False)
print_binned_split(df_full["hdl_group"], "HDL (mg/dL)")

tc_bins = [0, 200, 240, 9999]
tc_labels = ["<200 Desirable", "200-239 Borderline", ">=240 High"]
df_full["tc_group"] = pd.cut(df_full["total cholesterol-estimated result"], bins=tc_bins, labels=tc_labels, right=False)
print_binned_split(df_full["tc_group"], "Total Cholesterol (mg/dL)")

# ──────────────────────────────────────────────────────────────
# 5. DAYS FROM REFERENCE (lab/measurement recency)
# ──────────────────────────────────────────────────────────────
day_bins = [-1, 0, 30, 90, 180, 365, 9999]
day_labels = ["0 Same Day", "1-30 Very Recent", "31-90 Recent",
              "91-180 Moderate", "181-365 Stale", ">365 Outdated"]

for col in [c for c in df_full.columns if "days from reference" in c]:
    binned = pd.cut(df_full[col], bins=day_bins, labels=day_labels, right=True)
    print_binned_split(binned, col)

# ──────────────────────────────────────────────────────────────
# 6. COMORBIDITY COUNTS
# ──────────────────────────────────────────────────────────────
dx_bins = [-1, 0, 2, 5, 9999]
dx_labels = ["0 None", "1-2", "3-5", "6+"]

for col in ["cad-count", "copd-count"]:
    binned = pd.cut(df_full[col], bins=dx_bins, labels=dx_labels, right=True)
    print_binned_split(binned, col)

# ──────────────────────────────────────────────────────────────
# 7. UTILIZATION
# ──────────────────────────────────────────────────────────────
util_bins = [-1, 0, 1, 3, 9, 9999]
util_labels = ["0 None", "1", "2-3", "4-9", "10+"]

for col in ["ed vist count-count", "pcp visit count-count", "admission count-count"]:
    binned = pd.cut(df_full[col], bins=util_bins, labels=util_labels, right=True)
    print_binned_split(binned, col)

# ──────────────────────────────────────────────────────────────
# 8. MEDICATION ORDERS
# ──────────────────────────────────────────────────────────────
med_bins = [-1, 0, 1, 3, 9999]
med_labels = ["0 None", "1", "2-3", "4+"]

for col in ["glp-1 orders-count", "insulin orders-count", "metformin orders-count",
            "sglt2 orders-count", "sulfonylurea orders-count", "dpp4 orders-count"]:
    binned = pd.cut(df_full[col], bins=med_bins, labels=med_labels, right=True)
    print_binned_split(binned, col)

# ──────────────────────────────────────────────────────────────
# 9. INSURANCE
# ──────────────────────────────────────────────────────────────
print_binned_split(df_full["payor first op visit-primary insurance plan"], "Primary Insurance Plan")

# ──────────────────────────────────────────────────────────────
# 10. ADI (Area Deprivation Index)
# ──────────────────────────────────────────────────────────────
# State rank (1-10 decile)
col = "adi-adi state rank"
numeric_adi_s = pd.to_numeric(df_full[col], errors="coerce")
state_bins = [0, 2, 4, 6, 8, 10]
state_labels = ["1-2 Least Deprived", "3-4", "5-6", "7-8", "9-10 Most Deprived"]
binned = pd.cut(numeric_adi_s, bins=state_bins, labels=state_labels, right=True)
suppressed = (df_full[col] == "P").sum() + df_full[col].isna().sum()
print_binned_split(binned, f"{col}  [Suppressed/Missing: {suppressed:,}]")

# National rank (1-100 percentile)
col = "adi-adi national rank"
numeric_adi_n = pd.to_numeric(df_full[col], errors="coerce")
adi_bins = [0, 20, 40, 60, 80, 100]
adi_labels = ["1-20 Least Deprived", "21-40", "41-60", "61-80", "81-100 Most Deprived"]
binned = pd.cut(numeric_adi_n, bins=adi_bins, labels=adi_labels, right=True)
suppressed = (df_full[col] == "P").sum() + df_full[col].isna().sum()
print_binned_split(binned, f"{col}  [Suppressed/Missing: {suppressed:,}]")

# Clean up temp columns
df_full.drop(columns=["age_group", "bmi_group", "bmi_raw", "ldl_group",
                       "hdl_group", "tc_group", "age"], inplace=True, errors="ignore")
print("\nDescriptive statistics complete.")

# ── CELL 3B: Filter to non-null 2025 dates for pre-cleaning EDA ──
ref_col = "a1c 2025-collection date-time-days from reference"
df_pre = df_full.dropna(subset=[ref_col]).copy()
outcome_pre = outcome

# Compute age and basic derived columns for plotting
df_pre["age"] = 2025 - df_pre["date of birth"]
a1c_result_cols = [f"a1c {i}-estimated result" for i in range(1, 6)]
a1c_day_cols = [f"a1c {i}-collection date-time-days from reference" for i in range(1, 6)]

# A1c summary stats for plotting
a1c_vals_pre = df_pre[a1c_result_cols]
df_pre["a1c_mean"] = a1c_vals_pre.mean(axis=1)
df_pre["a1c_max"] = a1c_vals_pre.max(axis=1)
df_pre["a1c_min"] = a1c_vals_pre.min(axis=1)
df_pre["a1c_range"] = df_pre["a1c_max"] - df_pre["a1c_min"]
df_pre["a1c_std"] = a1c_vals_pre.std(axis=1)
df_pre["n_a1c_visits"] = a1c_vals_pre.notna().sum(axis=1)
df_pre["a1c_delta_1_2"] = df_pre["a1c 2-estimated result"] - df_pre["a1c 1-estimated result"]
df_pre["bmi_raw"] = df_pre["weight-estimated result"] / ((df_pre["height-estimated result"] / 100) ** 2)
df_pre["adi_national_rank"] = pd.to_numeric(df_pre.get("adi-adi national rank", pd.Series(dtype=float)), errors="coerce")
df_pre["adi_state_rank"] = pd.to_numeric(df_pre.get("adi-adi state rank", pd.Series(dtype=float)), errors="coerce")

y_pre = df_pre[outcome_pre].astype(int)
ctrl_pre = df_pre[df_pre[outcome_pre] == False]
unctrl_pre = df_pre[df_pre[outcome_pre] == True]

print(f"Pre-cleaning cohort: {len(df_pre):,} patients")
print(f"Uncontrolled: {unctrl_pre.shape[0]:,} ({y_pre.mean()*100:.1f}%)")

# ── CELL 3C: Missingness bar chart ──
miss_cols = {
    "A1C Visit 1": "a1c 1-estimated result", "A1C Visit 2": "a1c 2-estimated result",
    "A1C Visit 3": "a1c 3-estimated result", "A1C Visit 4": "a1c 4-estimated result",
    "A1C Visit 5": "a1c 5-estimated result",
    "Weight": "weight-estimated result", "Height": "height-estimated result",
    "LDL": "ldl-estimated result", "HDL": "hdl-estimated result",
    "Total Chol": "total cholesterol-estimated result",
    "Insurance": "payor first op visit-primary insurance plan",
    "ADI State": "adi-adi state rank", "ADI National": "adi-adi national rank",
    "Gender": "gender at birth", "Ethnicity": "ethnicity", "Race": "race - primary",
}
# Compute BMI missingness from weight/height
miss_pcts = {}
for label, col in miss_cols.items():
    miss_pcts[label] = df_pre[col].isna().mean() * 100
miss_pcts["BMI (derived)"] = ((df_pre["weight-estimated result"].isna()) | (df_pre["height-estimated result"].isna())).mean() * 100

labels = list(miss_pcts.keys())
values = list(miss_pcts.values())

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(labels, values, color="#3b82f6", alpha=0.85, edgecolor="white")
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f"{val:.1f}%", va="center", fontsize=9)
ax.set_xlabel("% Missing")
ax.set_title(f"Percentage of Missing Values (N={len(df_pre):,})", fontweight="bold")
ax.invert_yaxis()
ax.grid(alpha=0.2, axis="x")
fig.tight_layout()
plt.show()

# ── CELL 3D: Demographics - % uncontrolled by group ──
C_CTRL = "#2196F3"; C_UNCT = "#E53935"

def plot_pct_uncontrolled(df_in, col, bins=None, labels_list=None, ax=None, title=""):
    if bins is not None:
        series = pd.cut(df_in[col], bins=bins, labels=labels_list, right=True)
    else:
        series = df_in[col]
    combined = pd.DataFrame({"group": series, "outcome": df_in[outcome_pre]})
    grouped = combined.groupby("group", observed=False)["outcome"]
    rates = grouped.mean() * 100
    counts = grouped.count()
    overall = df_in[outcome_pre].mean() * 100
    bars = ax.bar(range(len(rates)), rates, color=C_UNCT, alpha=0.85, edgecolor="white")
    for j, (bar, rate) in enumerate(zip(bars, rates)):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{rate:.1f}%", ha="center", va="bottom", fontsize=8, fontweight=500)
    ax.axhline(overall, color="gray", ls="--", lw=1, alpha=0.6, label=f"Overall ({overall:.1f}%)")
    ax.set_xticks(range(len(rates)))
    ax.set_xticklabels(rates.index, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("% Uncontrolled")
    ax.set_title(title, fontweight="bold")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.2, axis="y")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
age_bins = [0, 29, 39, 49, 59, 69, 79, 200]
age_labels = ["18-29", "30-39", "40-49", "50-59", "60-69", "70-79", "80+"]
df_pre["age_group"] = pd.cut(df_pre["age"], bins=age_bins, labels=age_labels, right=True)
plot_pct_uncontrolled(df_pre, "age_group", ax=axes[0,0], title="Age Group")
plot_pct_uncontrolled(df_pre, "gender at birth", ax=axes[0,1], title="Gender")
plot_pct_uncontrolled(df_pre, "race - primary", ax=axes[1,0], title="Race")
plot_pct_uncontrolled(df_pre, "ethnicity", ax=axes[1,1], title="Ethnicity")
fig.suptitle("Demographics: % Uncontrolled by Group", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

# ── CELL 3E: A1c at each visit - % uncontrolled by ADA bin ──
a1c_bins = [0, 5.7, 6.5, 7.0, 8.0, 9.0, 100]
a1c_labels_plot = ["<5.7", "5.7-6.4", "6.5-7.0", "7.1-8.0", "8.1-9.0", ">9.0"]

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
for i, (col, ax) in enumerate(zip(a1c_result_cols, axes)):
    valid = df_pre[[col, outcome_pre]].dropna(subset=[col])
    valid["bin"] = pd.cut(valid[col], bins=a1c_bins, labels=a1c_labels_plot, right=True)
    rates = valid.groupby("bin", observed=False)[outcome_pre].mean() * 100
    overall = valid[outcome_pre].mean() * 100
    n = len(valid)
    bars = ax.bar(range(len(rates)), rates, color=C_UNCT, alpha=0.85, edgecolor="white")
    for bar, rate in zip(bars, rates):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f"{rate:.1f}%", ha="center", va="bottom", fontsize=8)
    ax.axhline(overall, color="gray", ls="--", lw=1, alpha=0.6, label=f"Overall ({overall:.1f}%)")
    ax.set_xticks(range(len(rates)))
    ax.set_xticklabels(a1c_labels_plot, rotation=45, ha="right", fontsize=8)
    ax.set_title(f"A1C Visit {i+1}\n(n={n:,})", fontweight="bold")
    ax.set_ylabel("% Uncontrolled") if i == 0 else None
    ax.legend(fontsize=6)
    ax.grid(alpha=0.2, axis="y")
fig.suptitle("A1C at Each Visit: % Uncontrolled Rate by Bin", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

# ── CELL 3F: A1c box & whisker at each visit by outcome ──
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes_flat = axes.flatten()

for i, col in enumerate(a1c_result_cols):
    ax = axes_flat[i]
    ctrl_vals = ctrl_pre[col].dropna()
    unctrl_vals = unctrl_pre[col].dropna()
    n_total = ctrl_vals.shape[0] + unctrl_vals.shape[0]
    bp = ax.boxplot([ctrl_vals, unctrl_vals], labels=["Controlled", "Uncontrolled"],
                    patch_artist=True, widths=0.5)
    for patch, c in zip(bp["boxes"], [C_CTRL, C_UNCT]):
        patch.set_facecolor(c); patch.set_alpha(0.5)
    ax.axhline(7.0, color="gray", ls="--", lw=0.8, alpha=0.5, label="ADA 7.0%")
    # Annotate medians
    ax.text(1, ctrl_vals.median(), f"med={ctrl_vals.median():.1f}", fontsize=8,
            ha="center", va="top", bbox=dict(boxstyle="round,pad=0.2", fc=C_CTRL, alpha=0.3))
    ax.text(2, unctrl_vals.median(), f"med={unctrl_vals.median():.1f}", fontsize=8,
            ha="center", va="top", bbox=dict(boxstyle="round,pad=0.2", fc=C_UNCT, alpha=0.3))
    ax.set_ylabel("A1C (%)")
    ax.set_title(f"A1C Visit {i+1} (n={n_total:,})", fontweight="bold")
    ax.legend(fontsize=7)

axes_flat[5].set_visible(False)
fig.suptitle("A1C Box & Whisker at Each Visit by Outcome", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

# ── CELL 3G: A1c trajectory summary stats by outcome ──
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

summary_cols = [("a1c_mean", "Mean A1C (%)"), ("a1c_max", "Max A1C (%)"),
                ("a1c_min", "Min A1C (%)"), ("a1c_range", "A1C Range (max-min)"),
                ("a1c_std", "A1C Std Dev"), ("n_a1c_visits", "# A1C Visits")]

for ax, (col, label) in zip(axes.flatten(), summary_cols):
    c_vals = ctrl_pre[col].dropna()
    u_vals = unctrl_pre[col].dropna()
    bp = ax.boxplot([c_vals, u_vals], labels=["Controlled", "Uncontrolled"],
                    patch_artist=True, widths=0.5)
    for patch, c in zip(bp["boxes"], [C_CTRL, C_UNCT]):
        patch.set_facecolor(c); patch.set_alpha(0.5)
    ax.text(1, c_vals.median(), f"med={c_vals.median():.1f}", fontsize=8, ha="center", va="top",
            bbox=dict(boxstyle="round,pad=0.2", fc=C_CTRL, alpha=0.3))
    ax.text(2, u_vals.median(), f"med={u_vals.median():.1f}", fontsize=8, ha="center", va="top",
            bbox=dict(boxstyle="round,pad=0.2", fc=C_UNCT, alpha=0.3))
    ax.set_ylabel(label.split("(")[0].strip())
    ax.set_title(f"{label} by Outcome", fontweight="bold")

fig.suptitle("A1C Trajectory Summary Statistics by Outcome", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

# ── CELL 3H: A1c threshold analysis ──
a1c_vals_pre = df_pre[a1c_result_cols]
n_vis = a1c_vals_pre.notna().sum(axis=1).clip(lower=1)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Mean # events above threshold
ax = axes[0, 0]
for thresh, t_str, offset in [(7.0, ">7.0", -0.15), (8.0, ">8.0", 0), (9.0, ">9.0", 0.15)]:
    cnt = (a1c_vals_pre > thresh).sum(axis=1)
    c_mean = cnt[y_pre == 0].mean()
    u_mean = cnt[y_pre == 1].mean()
    x = np.array([0, 1])
    ax.bar(x + offset, [c_mean, u_mean], width=0.14, label=t_str)
    for xi, val in zip(x + offset, [c_mean, u_mean]):
        ax.text(xi, val + 0.02, f"{val:.2f}", ha="center", fontsize=7)
ax.set_xticks([0, 1]); ax.set_xticklabels(["Controlled", "Uncontrolled"])
ax.set_ylabel("Mean # Visits Above Threshold")
ax.set_title("Mean # A1C Events Above Threshold", fontweight="bold")
ax.legend(); ax.grid(alpha=0.2, axis="y")

# Panel 2: Mean % above threshold
ax = axes[0, 1]
for thresh, t_str, offset in [(7.0, ">7.0", -0.15), (8.0, ">8.0", 0), (9.0, ">9.0", 0.15)]:
    pct = (a1c_vals_pre > thresh).sum(axis=1) / n_vis * 100
    c_mean = pct[y_pre == 0].mean()
    u_mean = pct[y_pre == 1].mean()
    x = np.array([0, 1])
    ax.bar(x + offset, [c_mean, u_mean], width=0.14, label=t_str)
    for xi, val in zip(x + offset, [c_mean, u_mean]):
        ax.text(xi, val + 0.5, f"{val:.1f}%", ha="center", fontsize=7)
ax.set_xticks([0, 1]); ax.set_xticklabels(["Controlled", "Uncontrolled"])
ax.set_ylabel("Mean % Visits Above Threshold")
ax.set_title("Mean % A1C Events Above Threshold", fontweight="bold")
ax.legend(); ax.grid(alpha=0.2, axis="y")

# Panel 3: % above threshold at each visit
ax = axes[1, 0]
visit_labels = [f"V{i+1}\n(n={a1c_vals_pre.iloc[:,i].notna().sum():,})" for i in range(5)]
for thresh, t_str, color, marker in [(7.0, ">7.0", "#f59e0b", "o"), (8.0, ">8.0", "#ef4444", "s"), (9.0, ">9.0", "#8b5cf6", "^")]:
    for grp, ls, m_style in [(0, "-", "o"), (1, "--", "s")]:
        pcts = []
        for j in range(5):
            valid = df_pre[[a1c_result_cols[j], outcome_pre]].dropna(subset=[a1c_result_cols[j]])
            grp_vals = valid[valid[outcome_pre] == grp][a1c_result_cols[j]]
            pcts.append((grp_vals > thresh).mean() * 100)
        label = f"{'Controlled' if grp==0 else 'Uncontrolled'}" if thresh == 7.0 else None
        ax.plot(range(5), pcts, color=color, ls=ls, marker=m_style, markersize=5, label=label)
ax.set_xticks(range(5)); ax.set_xticklabels(visit_labels, fontsize=8)
ax.set_ylabel("% Above Threshold")
ax.set_title("% Above Threshold at Each Visit", fontweight="bold")
ax.legend(fontsize=7); ax.grid(alpha=0.2)

# Panel 4: Distribution of # events > 7.0
ax = axes[1, 1]
cnt_above = (a1c_vals_pre > 7.0).sum(axis=1)
for grp, color, label in [(0, C_CTRL, "Controlled"), (1, C_UNCT, "Uncontrolled")]:
    vals = cnt_above[y_pre == grp]
    counts = vals.value_counts(normalize=True).sort_index() * 100
    ax.bar(counts.index + (0.15 if grp == 1 else -0.15), counts.values, width=0.3,
           color=color, alpha=0.8, label=label)
ax.set_xlabel("# Visits with A1C > 7.0"); ax.set_ylabel("% of Patients")
ax.set_title("Distribution of # Events > 7.0", fontweight="bold")
ax.legend(); ax.grid(alpha=0.2, axis="y")

fig.suptitle("A1C Threshold Analysis", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

# ── CELL 3I: Clinical measures, comorbidities, utilization, meds, social ──
# Clinical measures % uncontrolled
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
bmi_bins = [0, 18.5, 25, 30, 35, 40, 200]; bmi_labels = ["<18.5", "18.5-24.9", "25-29.9", "30-34.9", "35-39.9", ">=40"]
df_pre["bmi_grp"] = pd.cut(df_pre["bmi_raw"], bins=bmi_bins, labels=bmi_labels, right=False)
plot_pct_uncontrolled(df_pre, "bmi_grp", ax=axes[0], title="BMI (WHO)")
ldl_bins = [0, 70, 100, 130, 160, 9999]; ldl_labels = ["<70", "70-99", "100-129", "130-159", ">=160"]
df_pre["ldl_grp"] = pd.cut(df_pre["ldl-estimated result"], bins=ldl_bins, labels=ldl_labels, right=False)
plot_pct_uncontrolled(df_pre, "ldl_grp", ax=axes[1], title="LDL")
hdl_bins = [0, 40, 60, 9999]; hdl_labels = ["<40", "40-59", ">=60"]
df_pre["hdl_grp"] = pd.cut(df_pre["hdl-estimated result"], bins=hdl_bins, labels=hdl_labels, right=False)
plot_pct_uncontrolled(df_pre, "hdl_grp", ax=axes[2], title="HDL")
tc_bins = [0, 200, 240, 9999]; tc_labels = ["<200", "200-239", ">=240"]
df_pre["tc_grp"] = pd.cut(df_pre["total cholesterol-estimated result"], bins=tc_bins, labels=tc_labels, right=False)
plot_pct_uncontrolled(df_pre, "tc_grp", ax=axes[3], title="Total Chol")
fig.suptitle("Clinical Measures: % Uncontrolled", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout(); plt.show()

# Comorbidities & utilization
fig, axes = plt.subplots(1, 5, figsize=(22, 5))
dx_bins = [-1, 0, 2, 5, 9999]; dx_labels = ["0", "1-2", "3-5", "6+"]
df_pre["cad_grp"] = pd.cut(df_pre["cad-count"], bins=dx_bins, labels=dx_labels, right=True)
plot_pct_uncontrolled(df_pre, "cad_grp", ax=axes[0], title="CAD")
df_pre["copd_grp"] = pd.cut(df_pre["copd-count"], bins=dx_bins, labels=dx_labels, right=True)
plot_pct_uncontrolled(df_pre, "copd_grp", ax=axes[1], title="COPD")
util_bins = [-1, 0, 1, 3, 9, 9999]; util_labels = ["0", "1", "2-3", "4-9", "10+"]
for col_name, title, ax_i in [("ed vist count-count", "ED Visits", 2),
                                ("pcp visit count-count", "PCP Visits", 3),
                                ("admission count-count", "Admissions", 4)]:
    df_pre[f"{title}_grp"] = pd.cut(df_pre[col_name], bins=util_bins, labels=util_labels, right=True)
    plot_pct_uncontrolled(df_pre, f"{title}_grp", ax=axes[ax_i], title=title)
fig.suptitle("Comorbidities & Utilization: % Uncontrolled", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout(); plt.show()

# Medication orders
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
med_bins = [-1, 0, 1, 3, 9999]; med_labels = ["0", "1", "2-3", "4+"]
for ax, (col_name, title) in zip(axes.flatten(), [
    ("glp-1 orders-count", "GLP-1"), ("insulin orders-count", "Insulin"),
    ("metformin orders-count", "Metformin"), ("sglt2 orders-count", "SGLT2"),
    ("sulfonylurea orders-count", "Sulfonylurea"), ("dpp4 orders-count", "DPP-4")]):
    df_pre[f"{title}_grp"] = pd.cut(df_pre[col_name], bins=med_bins, labels=med_labels, right=True)
    plot_pct_uncontrolled(df_pre, f"{title}_grp", ax=ax, title=f"{title} Orders")
fig.suptitle("Medication Orders: % Uncontrolled", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout(); plt.show()

# Social determinants
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
state_bins = [0, 2, 4, 6, 8, 10]; state_labels = ["1-2", "3-4", "5-6", "7-8", "9-10"]
df_pre["adi_s_grp"] = pd.cut(df_pre["adi_state_rank"], bins=state_bins, labels=state_labels, right=True)
plot_pct_uncontrolled(df_pre, "adi_s_grp", ax=axes[0], title="ADI State (Decile)")
adi_bins = [0, 20, 40, 60, 80, 100]; adi_labels = ["1-20", "21-40", "41-60", "61-80", "81-100"]
df_pre["adi_n_grp"] = pd.cut(df_pre["adi_national_rank"], bins=adi_bins, labels=adi_labels, right=True)
plot_pct_uncontrolled(df_pre, "adi_n_grp", ax=axes[1], title="ADI National")
# Insurance: simplified
ins_map = {"Medicare": "Medicare", "Medicare Managed Care": "Medicare MC",
    "Medicaid": "Medicaid", "Medicaid Managed Care": "Medicaid MC",
    "Managed Care, Unspecified": "Managed Care", "Blue Cross Managed Care": "BCBS MC",
    "Managed Care Preferred Provider Organization (PPO)": "PPO",
    "Private Health Insurance": "Private", "No matching concept": "No Match",
    "pending review": "Pending", "Department of Veterans Affairs": "VA",
    "DoD TRICARE Prime - Health Management Organization (HMO)": "TRICARE"}
df_pre["ins_short"] = df_pre["payor first op visit-primary insurance plan"].map(ins_map)
plot_pct_uncontrolled(df_pre.dropna(subset=["ins_short"]), "ins_short", ax=axes[2], title="Insurance Type")
fig.suptitle("Social Determinants: % Uncontrolled", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout(); plt.show()

# ── CELL 3J: Days from reference distributions by outcome ──
date_cols = [(f"a1c {i}-collection date-time-days from reference", f"A1C Visit {i}") for i in range(1, 6)]
date_cols.append(("a1c 2025-collection date-time-days from reference", "A1C 2025 (Outcome)"))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, (col, title) in zip(axes.flatten(), date_cols):
    valid = df_pre[col].dropna()
    c_vals = ctrl_pre[col].dropna()
    u_vals = unctrl_pre[col].dropna()
    ax.hist(c_vals, bins=50, alpha=0.5, color=C_CTRL, density=True, label="Controlled")
    ax.hist(u_vals, bins=50, alpha=0.5, color=C_UNCT, density=True, label="Uncontrolled")
    ax.set_xlabel("Days from Reference")
    ax.set_ylabel("Density")
    ax.set_title(f"{title}\n(n={len(valid):,})", fontweight="bold")
    ax.legend(fontsize=7)
fig.suptitle("Days from Reference: Distribution by Outcome", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout(); plt.show()

# ── CELL 3K: A1c trajectory plots (by age, race, gender, baseline A1c) ──
# A1c trajectory by age group
fig, ax = plt.subplots(figsize=(12, 8))
age_groups_traj = ["18-29", "30-39", "40-49", "50-59", "60-69", "70-79", "80+"]
colors_age = plt.cm.tab10(np.linspace(0, 1, len(age_groups_traj)))
for j, ag in enumerate(age_groups_traj):
    for grp, ls, lw in [(False, "-", 1.5), (True, "--", 1.5)]:
        mask = (df_pre["age_group"] == ag) & (df_pre[outcome_pre] == grp)
        means = [df_pre.loc[mask, c].mean() for c in a1c_result_cols]
        label = f"{ag} {'Unctrl' if grp else 'Ctrl'}"
        ax.plot(range(1, 6), means, color=colors_age[j], ls=ls, lw=lw, marker="o" if not grp else "s",
                markersize=4, label=label)
ax.axhline(7.0, color="gray", ls=":", lw=0.8, alpha=0.5)
ax.set_xlabel("A1C Visit Number"); ax.set_ylabel("Mean A1C (%)")
ax.set_title("Mean A1C Trajectory by Age Group", fontweight="bold")
ax.legend(fontsize=6, ncol=4, loc="lower right")
ax.grid(alpha=0.2)
fig.suptitle("A1C Over Time: Age Groups (Solid=Controlled, Dashed=Uncontrolled)",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout(); plt.show()

# A1c trajectory by race and gender
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
ax = axes[0]
for race, color in [("White", "#2563eb"), ("Black or African American", "#9a3412"), ("Asian", "#0d9488")]:
    for grp, ls in [(False, "-"), (True, "--")]:
        mask = (df_pre["race - primary"] == race) & (df_pre[outcome_pre] == grp)
        means = [df_pre.loc[mask, c].mean() for c in a1c_result_cols]
        ax.plot(range(1, 6), means, color=color, ls=ls, lw=2, marker="o" if not grp else "s",
                markersize=5, label=f"{race.split()[0]} {'Unctrl' if grp else 'Ctrl'}")
ax.axhline(7.0, color="gray", ls=":", lw=0.8); ax.set_xlabel("A1C Visit Number"); ax.set_ylabel("Mean A1C (%)")
ax.set_title("By Race (Top 3)", fontweight="bold"); ax.legend(fontsize=7); ax.grid(alpha=0.2)

ax = axes[1]
for gender, color in [("FEMALE", "#2563eb"), ("MALE", "#0d9488")]:
    for grp, ls in [(False, "-"), (True, "--")]:
        mask = (df_pre["gender at birth"] == gender) & (df_pre[outcome_pre] == grp)
        means = [df_pre.loc[mask, c].mean() for c in a1c_result_cols]
        ax.plot(range(1, 6), means, color=color, ls=ls, lw=2, marker="o" if not grp else "s",
                markersize=5, label=f"{gender} {'Unctrl' if grp else 'Ctrl'}")
ax.axhline(7.0, color="gray", ls=":", lw=0.8); ax.set_xlabel("A1C Visit Number"); ax.set_ylabel("Mean A1C (%)")
ax.set_title("By Gender", fontweight="bold"); ax.legend(fontsize=7); ax.grid(alpha=0.2)
fig.suptitle("A1C Trajectory by Demographics (Solid=Ctrl, Dashed=Unctrl)",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout(); plt.show()

# A1c trajectory by baseline A1c category
fig, ax = plt.subplots(figsize=(12, 8))
a1c_cats = ["<5.7", "5.7-6.4", "6.5-7.0", "7.1-8.0", "8.1-9.0", ">9.0"]
df_pre["a1c_v1_bin"] = pd.cut(df_pre["a1c 1-estimated result"], bins=a1c_bins, labels=a1c_cats, right=True)
colors_a1c = plt.cm.viridis(np.linspace(0, 0.9, len(a1c_cats)))
for j, cat in enumerate(a1c_cats):
    for grp, ls in [(False, "-"), (True, "--")]:
        mask = (df_pre["a1c_v1_bin"] == cat) & (df_pre[outcome_pre] == grp)
        if mask.sum() < 10: continue
        means = [df_pre.loc[mask, c].mean() for c in a1c_result_cols]
        ax.plot(range(1, 6), means, color=colors_a1c[j], ls=ls, lw=2, marker="o" if not grp else "s",
                markersize=4, label=f"{cat} {'Unctrl' if grp else 'Ctrl'}")
ax.axhline(7.0, color="gray", ls=":", lw=0.8); ax.set_xlabel("A1C Visit Number"); ax.set_ylabel("Mean A1C (%)")
ax.set_title("Mean A1C Trajectory by Baseline A1C Category", fontweight="bold")
ax.legend(fontsize=6, ncol=4, loc="center right"); ax.grid(alpha=0.2)
fig.suptitle("A1C Over Time: Stratified by Visit 1 A1C (Solid=Ctrl, Dashed=Unctrl)",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout(); plt.show()

# ── CELL 3L: A1c trajectory by comorbidity, utilization, meds, ADI ──
def plot_trajectory_stratified(df_in, group_col, group_labels, title, ax):
    colors_t = plt.cm.tab10(np.linspace(0, 1, len(group_labels)))
    for j, grp_label in enumerate(group_labels):
        for out, ls in [(False, "-"), (True, "--")]:
            mask = (df_in[group_col] == grp_label) & (df_in[outcome_pre] == out)
            if mask.sum() < 10: continue
            means = [df_in.loc[mask, c].mean() for c in a1c_result_cols]
            label = f"{grp_label} {'Unctrl' if out else 'Ctrl'}"
            ax.plot(range(1, 6), means, color=colors_t[j], ls=ls, lw=1.5,
                    marker="o" if not out else "s", markersize=4, label=label)
    ax.axhline(7.0, color="gray", ls=":", lw=0.8, alpha=0.5)
    ax.set_xlabel("A1C Visit Number"); ax.set_ylabel("Mean A1C (%)")
    ax.set_title(f"By {title}", fontweight="bold")
    ax.legend(fontsize=5, ncol=2); ax.grid(alpha=0.2)

# Comorbidity
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
plot_trajectory_stratified(df_pre, "cad_grp", dx_labels, "CAD Burden", axes[0])
plot_trajectory_stratified(df_pre, "copd_grp", dx_labels, "COPD Burden", axes[1])
fig.suptitle("A1C Trajectory by Comorbidity Burden (Solid=Ctrl, Dashed=Unctrl)",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout(); plt.show()

# Utilization
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_trajectory_stratified(df_pre, "ED Visits_grp", util_labels, "ED Visits", axes[0])
plot_trajectory_stratified(df_pre, "PCP Visits_grp", util_labels, "PCP Visits", axes[1])
plot_trajectory_stratified(df_pre, "Admissions_grp", util_labels, "Admissions", axes[2])
fig.suptitle("A1C Trajectory by Utilization (Solid=Ctrl, Dashed=Unctrl)",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout(); plt.show()

# Medications
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, title in zip(axes.flatten(), ["GLP-1", "Insulin", "Metformin", "SGLT2", "Sulfonylurea", "DPP-4"]):
    plot_trajectory_stratified(df_pre, f"{title}_grp", med_labels, f"{title} Orders", ax)
fig.suptitle("A1C Trajectory by Medication Orders (Solid=Ctrl, Dashed=Unctrl)",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout(); plt.show()

# ADI
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
plot_trajectory_stratified(df_pre, "adi_s_grp", state_labels, "ADI State Decile", axes[0])
plot_trajectory_stratified(df_pre, "adi_n_grp", adi_labels, "ADI National Quintile", axes[1])
fig.suptitle("A1C Trajectory by Area Deprivation (Solid=Ctrl, Dashed=Unctrl)",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout(); plt.show()

# ── CELL 3M: Correlation matrix + preliminary feature importance + PDPs ──
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.inspection import PartialDependenceDisplay

# Correlation matrix
corr_cols = ["age", "a1c 1-estimated result", "a1c 2-estimated result", "bmi_raw",
    "ldl-estimated result", "hdl-estimated result", "total cholesterol-estimated result",
    "cad-count", "copd-count", "ed vist count-count", "pcp visit count-count",
    "admission count-count", "glp-1 orders-count", "insulin orders-count",
    "metformin orders-count", "sglt2 orders-count", "sulfonylurea orders-count",
    "dpp4 orders-count", "adi_national_rank"]
corr_labels = ["Age", "A1C-1", "A1C-2", "BMI", "LDL", "HDL", "TotalChol",
    "CAD", "COPD", "ED", "PCP", "Admits", "GLP1", "Insulin", "Metformin",
    "SGLT2", "Sulf", "DPP4", "ADI-Nat"]

valid_cols = [c for c in corr_cols if c in df_pre.columns]
valid_labels = [corr_labels[i] for i, c in enumerate(corr_cols) if c in df_pre.columns]
corr_df = df_pre[valid_cols + [outcome_pre]].copy()
corr_df[outcome_pre] = corr_df[outcome_pre].astype(int)
corr_df.columns = valid_labels + ["Uncontrolled"]
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, ax=ax, annot_kws={"size": 6}, linewidths=0.5,
            vmin=-1, vmax=1)
ax.set_title("Correlation Matrix (Numeric Features + Outcome)", fontweight="bold", fontsize=14)
fig.tight_layout(); plt.show()

# Preliminary gradient boosting feature importance
print("\nTraining preliminary gradient boosting for feature importance and PDPs...")
feat_cols_prelim = [c for c in valid_cols if c != outcome_pre]
feat_labels_prelim = [l for l, c in zip(valid_labels, valid_cols) if c != outcome_pre]

X_prelim = df_pre[feat_cols_prelim].copy()
from sklearn.impute import SimpleImputer
imp = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imp.fit_transform(X_prelim), columns=feat_cols_prelim, index=X_prelim.index)

gb = GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                                 subsample=0.8, random_state=42)
gb.fit(X_imp, y_pre)

importances = gb.feature_importances_
sorted_idx = np.argsort(importances)
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh([feat_labels_prelim[i] for i in sorted_idx],
        importances[sorted_idx], color="#3b82f6", alpha=0.85)
ax.set_xlabel("Feature Importance (Gini)")
ax.set_title("Gradient Boosting Feature Importance", fontweight="bold")
ax.grid(alpha=0.2, axis="x")
fig.tight_layout(); plt.show()

# Partial dependence plots - top 6
top_6_idx = np.argsort(-importances)[:6]
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, feat_i in zip(axes.flatten(), top_6_idx):
    PartialDependenceDisplay.from_estimator(gb, X_imp, [feat_i], ax=ax,
        feature_names=feat_labels_prelim, grid_resolution=30)
    ax.set_title(feat_labels_prelim[feat_i], fontweight="bold")
fig.suptitle("Partial Dependence Plots (Top 6 Features)", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout(); plt.show()

# PDPs - remaining features
remaining_idx = np.argsort(-importances)[6:]
n_remaining = len(remaining_idx)
n_cols_r = 3
n_rows_r = int(np.ceil(n_remaining / n_cols_r))
fig, axes = plt.subplots(n_rows_r, n_cols_r, figsize=(16, 4 * n_rows_r))
axes_flat = axes.flatten() if n_remaining > 1 else [axes]
for ax, feat_i in zip(axes_flat[:n_remaining], remaining_idx):
    PartialDependenceDisplay.from_estimator(gb, X_imp, [feat_i], ax=ax,
        feature_names=feat_labels_prelim, grid_resolution=30)
    ax.set_title(feat_labels_prelim[feat_i], fontweight="bold")
for ax in axes_flat[n_remaining:]:
    ax.set_visible(False)
fig.suptitle("Partial Dependence Plots (Remaining Features)", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout(); plt.show()

# Clean up pre-cleaning temp columns
df_pre = None  # free memory
print("Pre-cleaning EDA complete.")

## Phases 2-3: Shared Cleaning and Feature Engineering Functions

These functions guarantee identical logic for training and prediction. The `drop_rows` flag controls whether patients with null 2025 dates or zero valid A1c visits are excluded (training) or retained (prediction).

**Cleaning steps:** A1c values outside [3, 20] to NaN, negative LDL to NaN, weight >300kg to NaN, height <100cm to NaN, BMI outside [12, 80] to NaN, demographic recoding (Refuse/Unknown collapsed), ADI numeric coercion.

**Feature engineering:** 25 features surviving permutation importance validation. Starts from 97 candidate features, reduced via the 0.0005 AUC-drop noise floor criterion.

In [ ]:
# ── CELL 4: Shared cleaning function ──
A1C_RESULT_COLS = [f"a1c {i}-estimated result" for i in range(1, 6)]
A1C_DAY_COLS = [f"a1c {i}-collection date-time-days from reference" for i in range(1, 6)]
A1C_DAY_COLS_LOOKBACK = [
    "a1c 1-collection date-time-days from reference",
    "a1c 2-collection date-time-days from reference",
]
REF_COL = "a1c 2025-collection date-time-days from reference"

def clean_features(raw, drop_rows=True, verbose=True):
    # Apply training-pipeline value cleaning.
    # drop_rows=True  : training behavior (drops null-outcome and zero-a1c patients)
    # drop_rows=False : prediction behavior (keeps every row, cleans values only)
    df_c = raw.copy()
    report = {}

    # Step 1: Drop rows missing 2025 outcome reference date (training only)
    if drop_rows:
        before = len(df_c)
        df_c = df_c.dropna(subset=[REF_COL])
        report["dropped_no_2025_date"] = before - len(df_c)
        if verbose:
            print(f"  Dropped {report['dropped_no_2025_date']:,} patients with null 2025 date")

    # Step 2: A1c outlier cleaning: values outside [3, 20] to NaN
    n_a1c_nulled = 0
    for res_col, day_col in zip(A1C_RESULT_COLS, A1C_DAY_COLS):
        if res_col in df_c.columns:
            bad = (df_c[res_col] > 20.0) | (df_c[res_col] < 3.0)
            n_a1c_nulled += bad.sum()
            df_c.loc[bad, res_col] = np.nan
            if day_col in df_c.columns:
                df_c.loc[df_c[res_col].isna(), day_col] = np.nan
    report["a1c_values_nulled"] = int(n_a1c_nulled)
    if verbose:
        print(f"  A1c values nulled (outside [3,20]): {n_a1c_nulled}")

    # Drop patients with zero valid A1c visits (training only)
    if drop_rows:
        before = len(df_c)
        n_valid = df_c[A1C_RESULT_COLS].notna().sum(axis=1)
        df_c = df_c[n_valid > 0]
        report["dropped_zero_valid_a1c"] = before - len(df_c)
        if verbose:
            print(f"  Dropped {report['dropped_zero_valid_a1c']:,} patients with 0 valid A1c")

    # Step 3: Negative LDL to NaN
    if "ldl-estimated result" in df_c.columns:
        neg = (df_c["ldl-estimated result"] < 0).sum()
        df_c.loc[df_c["ldl-estimated result"] < 0, "ldl-estimated result"] = np.nan
        report["ldl_negatives_nulled"] = int(neg)

    # Step 4: Anthropometric outliers
    if "weight-estimated result" in df_c.columns:
        bad = (df_c["weight-estimated result"] > 300).sum()
        df_c.loc[df_c["weight-estimated result"] > 300, "weight-estimated result"] = np.nan
        report["weight_over_300"] = int(bad)
    if "height-estimated result" in df_c.columns:
        bad = (df_c["height-estimated result"] < 100).sum()
        df_c.loc[df_c["height-estimated result"] < 100, "height-estimated result"] = np.nan
        report["height_under_100"] = int(bad)

    # BMI computation and outlier filtering
    df_c["bmi"] = df_c["weight-estimated result"] / ((df_c["height-estimated result"] / 100) ** 2)
    bad_bmi = ((df_c["bmi"] < 12) | (df_c["bmi"] > 80)).sum()
    df_c.loc[(df_c["bmi"] < 12) | (df_c["bmi"] > 80), "bmi"] = np.nan
    report["bmi_outliers"] = int(bad_bmi)

    # Step 5: Demographic recoding
    df_c["gender at birth"] = df_c["gender at birth"].replace(
        {"Gender unknown": "Unknown/Not Reported"}).fillna("Unknown/Not Reported")
    df_c["race - primary"] = df_c["race - primary"].replace({
        "Refuse to answer": "Unknown/Not Reported",
        "Unknown racial group": "Unknown/Not Reported",
    }).fillna("Unknown/Not Reported")
    df_c["ethnicity"] = df_c["ethnicity"].replace({
        "Refusal by patient to provide information about ethnic group": "Unknown/Not Reported",
        "Patient ethnicity unknown": "Unknown/Not Reported",
    }).fillna("Unknown/Not Reported")

    # Step 7: ADI cleanup
    df_c["adi_national_rank"] = pd.to_numeric(
        df_c.get("adi-adi national rank", pd.Series(dtype=float)), errors="coerce")

    # Step 10: Age
    df_c["age"] = 2025 - df_c["date of birth"]

    if verbose:
        print(f"  Final cohort: {len(df_c):,} patients")

    return df_c, report

In [ ]:
# ── CELL 5: Shared feature engineering function ──
def engineer_features(df_c, training_race_vocab=None, verbose=True):
    # Apply feature engineering to a cleaned dataframe.
    # training_race_vocab: required at prediction time so unseen categories
    # map to 'Unknown/Not Reported'. Pass None at training time.
    # Returns: (df_engineered, race_vocab)
    df_c = df_c.copy()

    a1c_vals = df_c[A1C_RESULT_COLS]
    n_visits = a1c_vals.notna().sum(axis=1).clip(lower=1)

    # A1c core stats
    df_c["a1c_mean"] = a1c_vals.mean(axis=1)
    df_c["a1c_max"]  = a1c_vals.max(axis=1)
    df_c["a1c_min"]  = a1c_vals.min(axis=1)
    df_c["a1c_std"]  = a1c_vals.std(axis=1)
    df_c["n_a1c_visits"] = n_visits

    # A1c delta and trend (unified definition: >= 1.0)
    df_c["a1c_delta_1_2"] = df_c["a1c 2-estimated result"] - df_c["a1c 1-estimated result"]
    df_c["a1c_trending_up"] = (df_c["a1c_delta_1_2"] >= 1.0).astype(float)
    df_c["a1c_trending_down"] = (df_c["a1c_delta_1_2"] <= -1.0).astype(float)
    df_c.loc[df_c["a1c_delta_1_2"].isna(), "a1c_trending_up"] = np.nan
    df_c.loc[df_c["a1c_delta_1_2"].isna(), "a1c_trending_down"] = np.nan

    # A1c threshold fractions (only 7.0 and 8.0 survived permutation importance)
    df_c["a1c_pct_above_70"] = (a1c_vals > 7.0).sum(axis=1) / n_visits * 100
    df_c["a1c_pct_above_80"] = (a1c_vals > 8.0).sum(axis=1) / n_visits * 100

    # Trajectory shape features
    day1 = df_c["a1c 1-collection date-time-days from reference"]
    day2 = df_c["a1c 2-collection date-time-days from reference"]
    val1 = df_c["a1c 1-estimated result"]
    val2 = df_c["a1c 2-estimated result"]
    time_diff_months = (day2 - day1) / 30.44

    df_c["a1c_slope_per_month"] = np.where(
        (time_diff_months.notna()) & (time_diff_months > 0),
        (val2 - val1) / time_diff_months, np.nan)

    first_valid_a1c = a1c_vals.bfill(axis=1).iloc[:, 0]
    last_valid_a1c  = a1c_vals.ffill(axis=1).iloc[:, -1]
    has_two = a1c_vals.notna().sum(axis=1) >= 2

    df_c["a1c_last_first_ratio"] = np.where(
        has_two & (first_valid_a1c > 0), last_valid_a1c / first_valid_a1c, np.nan)

    df_c["a1c_most_recent"] = last_valid_a1c
    df_c["days_v1_to_v2"] = day2 - day1

    # Gap to 2025 outcome
    last_lookback_day = df_c[A1C_DAY_COLS_LOOKBACK].max(axis=1)
    df_c["gap_to_2025"] = (df_c[REF_COL] - last_lookback_day)
    df_c.loc[df_c["gap_to_2025"] < 0, "gap_to_2025"] = 0

    # Visits per month (v8.1 fix: require >= 3 months observation span)
    obs_span_months = (last_lookback_day - day1) / 30.44
    df_c["visits_per_month"] = np.where(
        obs_span_months >= 3,
        n_visits / obs_span_months,
        np.nan
    )

    # Persistent moderate: mean 7.0-9.0 without upward trend
    df_c["persistent_moderate"] = (
        (df_c["a1c_mean"] >= 7.0) & (df_c["a1c_mean"] < 9.0) &
        (df_c["a1c_trending_up"].fillna(0) == 0)
    ).astype(int)

    # Medication intensity
    med_cols = ["glp-1 orders-count", "insulin orders-count", "metformin orders-count",
                "sglt2 orders-count", "sulfonylurea orders-count", "dpp4 orders-count"]
    total_med_orders = df_c[med_cols].sum(axis=1)
    df_c["orders_per_visit"] = total_med_orders / df_c["n_a1c_visits"].replace(0, np.nan)

    # Young elevated A1c (interaction: excess A1c / age)
    df_c["young_elevated_a1c"] = (
        (df_c["a1c_mean"] - 7.0).clip(lower=0) / df_c["age"].replace(0, np.nan)
    )

    # Race encoding (fold-safe vocabulary)
    race_vocab = training_race_vocab
    if race_vocab is None:
        race_vocab = sorted(df_c["race - primary"].dropna().unique().tolist())
    unseen = ~df_c["race - primary"].isin(race_vocab)
    df_c.loc[unseen, "race - primary"] = "Unknown/Not Reported"
    df_c["race - primary_enc"] = (
        df_c["race - primary"].astype("category")
        .cat.set_categories(race_vocab).cat.codes
    )

    if verbose:
        print(f"  Engineered features complete. Shape: {df_c.shape}")

    return df_c, race_vocab


# Final feature list (25 features, validated by permutation importance)
MINIMAL_FEATURES = [
    # Demographics
    "age", "race - primary_enc",
    # A1c core stats
    "a1c_mean", "a1c_max", "a1c_min", "a1c_std",
    "a1c 1-estimated result", "a1c_most_recent",
    # A1c threshold fractions
    "a1c_pct_above_70", "a1c_pct_above_80",
    # A1c trajectory
    "a1c_slope_per_month", "a1c_last_first_ratio", "days_v1_to_v2",
    "persistent_moderate",
    # Timing / cadence
    "gap_to_2025", "visits_per_month",
    # Medications (3 of 6 drug classes survived)
    "insulin orders-count", "metformin orders-count", "sulfonylurea orders-count",
    "orders_per_visit",
    # Engineered
    "young_elevated_a1c",
    # Other labs
    "bmi", "ldl-estimated result", "total cholesterol-estimated result",
    # Social determinants
    "adi_national_rank",
]
CAT_FEATURE_NAMES = ["race - primary_enc"]

print(f"Feature set defined: {len(MINIMAL_FEATURES)} features")
print(f"Categorical features: {CAT_FEATURE_NAMES}")

## Phase 4: Build the Filtered Training Cohort

Apply cleaning with `drop_rows=True` to produce the 35,794-patient cohort used for all model training and validation.

In [ ]:
# ── CELL 6: Build training cohort ──
print("="*70)
print("BUILDING TRAINING COHORT")
print("="*70)

df_clean_train, clean_report = clean_features(df_full, drop_rows=True, verbose=True)
df_feat_train, training_race_vocab = engineer_features(df_clean_train, verbose=True)

y = df_feat_train[outcome].astype(int)
X = df_feat_train[MINIMAL_FEATURES].copy()
cat_idx = [MINIMAL_FEATURES.index(c) for c in CAT_FEATURE_NAMES]

# Stratification key: outcome x visit group (balances both per fold)
df_feat_train["_visit_group"] = (df_feat_train["n_a1c_visits"] >= 2).astype(int)
strat_key = y.astype(str) + "_" + df_feat_train["_visit_group"].astype(str)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"\nTraining cohort: {len(y):,} patients")
print(f"Uncontrolled: {y.sum():,} ({y.mean()*100:.1f}%)")
print(f"Features: {X.shape[1]}")

## Phase 4B: Post-Cleaning EDA Figures (N=35,794)

Visual EDA on the fully cleaned and feature-engineered cohort. Demographics and A1c with cleaned categories, red flag subgroup analysis, 2025 measurement recency, correlation matrix with engineered features, feature importance, and partial dependence plots.

In [ ]:
# ── CELL 6C: Post-cleaning EDA - demographics and A1c ──
C_CTRL = "#2196F3"; C_UNCT = "#E53935"
ctrl_post = df_feat_train[df_feat_train[outcome] == False]
unctrl_post = df_feat_train[df_feat_train[outcome] == True]

def plot_pct_unctrl_post(df_in, col, ax, title="", bins=None, labels_list=None):
    if bins is not None:
        series = pd.cut(df_in[col], bins=bins, labels=labels_list, right=True)
    else:
        series = df_in[col]
    combined = pd.DataFrame({"group": series, "outcome": df_in[outcome]})
    grouped = combined.groupby("group", observed=False)["outcome"]
    rates = grouped.mean() * 100
    overall = df_in[outcome].mean() * 100
    bars = ax.bar(range(len(rates)), rates, color=C_UNCT, alpha=0.85, edgecolor="white")
    for bar, rate in zip(bars, rates):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{rate:.1f}%", ha="center", va="bottom", fontsize=8, fontweight=500)
    ax.axhline(overall, color="gray", ls="--", lw=1, alpha=0.6, label=f"Overall ({overall:.1f}%)")
    ax.set_xticks(range(len(rates)))
    ax.set_xticklabels(rates.index, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("% Uncontrolled"); ax.set_title(title, fontweight="bold")
    ax.legend(fontsize=7); ax.grid(alpha=0.2, axis="y")

# Demographics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
age_bins = [0, 29, 39, 49, 59, 69, 79, 200]
age_labels_post = ["18-29", "30-39", "40-49", "50-59", "60-69", "70-79", "80+"]
df_feat_train["age_group_post"] = pd.cut(df_feat_train["age"], bins=age_bins, labels=age_labels_post, right=True)
plot_pct_unctrl_post(df_feat_train, "age_group_post", axes[0,0], "Age Group")
plot_pct_unctrl_post(df_feat_train, "race - primary", axes[0,1], "Race")
plot_pct_unctrl_post(df_feat_train, "gender at birth", axes[1,0], "Gender")
plot_pct_unctrl_post(df_feat_train, "ethnicity", axes[1,1], "Ethnicity")
fig.suptitle("Demographics: % Uncontrolled by Group", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout(); plt.show()

# A1c at each visit with annotated counts
a1c_bins_post = [0, 5.7, 6.5, 7.0, 8.0, 9.0, 100]
a1c_labels_post = ["<5.7", "5.7-6.4", "6.5-7.0", "7.1-8.0", "8.1-9.0", ">9.0"]

fig, axes = plt.subplots(1, 5, figsize=(22, 6))
for i, (col, ax) in enumerate(zip(A1C_RESULT_COLS, axes)):
    valid = df_feat_train[[col, outcome]].dropna(subset=[col])
    valid["bin"] = pd.cut(valid[col], bins=a1c_bins_post, labels=a1c_labels_post, right=True)
    c_counts = valid[valid[outcome]==False].groupby("bin", observed=False).size()
    u_counts = valid[valid[outcome]==True].groupby("bin", observed=False).size()
    n_c = (valid[outcome]==False).sum(); n_u = (valid[outcome]==True).sum()
    c_pct = c_counts / n_c * 100; u_pct = u_counts / n_u * 100
    x = np.arange(len(a1c_labels_post)); width = 0.35
    bars_c = ax.bar(x - width/2, c_pct, width, color=C_CTRL, alpha=0.7, label=f"Ctrl (n={n_c:,})")
    bars_u = ax.bar(x + width/2, u_pct, width, color=C_UNCT, alpha=0.7, label=f"Unctrl (n={n_u:,})")
    for bar, cnt in zip(bars_c, c_counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f"{cnt:,}", ha="center", fontsize=6, color="#333")
    for bar, cnt in zip(bars_u, u_counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f"{cnt:,}", ha="center", fontsize=6, color="#333")
    ax.set_xticks(x); ax.set_xticklabels(a1c_labels_post, rotation=45, ha="right", fontsize=7)
    ax.set_ylabel("% within group"); ax.set_title(f"A1C Visit {i+1} (n={len(valid):,})", fontweight="bold")
    ax.legend(fontsize=6)
fig.suptitle("A1C at Each Visit: Distribution by Outcome (counts annotated)",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout(); plt.show()

# ── CELL 6D: Post-cleaning - Red flag subgroup analysis ──
print("="*70)
print("RED FLAG SUBGROUP ANALYSIS")
print("="*70)

subgroups = {
    "High Max + Recovering\n(max>9, delta<=-1)":
        (df_feat_train["a1c_max"] > 9) & (df_feat_train["a1c_delta_1_2"] <= -1),
    "High Max + Worsening\n(max>9, delta>=+1)":
        (df_feat_train["a1c_max"] > 9) & (df_feat_train["a1c_delta_1_2"] >= 1),
    "Controlled Start + Rising\n(V1<7, delta>=+1)":
        (df_feat_train["a1c 1-estimated result"] < 7) & (df_feat_train["a1c_delta_1_2"] >= 1),
    "High A1C + No Meds\n(max>8, 0 med orders)":
        (df_feat_train["a1c_max"] > 8) & (df_feat_train[["glp-1 orders-count", "insulin orders-count",
        "metformin orders-count", "sglt2 orders-count", "sulfonylurea orders-count",
        "dpp4 orders-count"]].sum(axis=1) == 0),
    "On Insulin + High Max\n(insulin>=2, max>9)":
        (df_feat_train["insulin orders-count"] >= 2) & (df_feat_train["a1c_max"] > 9),
    "High Variability\n(std>1.5, 2+ visits)":
        (df_feat_train["a1c_std"] > 1.5) & (df_feat_train["n_a1c_visits"] >= 2),
    "Low Engagement + High A1C\n(1 visit, A1C>8)":
        (df_feat_train["n_a1c_visits"] == 1) & (df_feat_train["a1c 1-estimated result"] > 8),
    "High Util + High A1C\n(ED>=3, max>8)":
        (df_feat_train["ed vist count-count"] >= 3) & (df_feat_train["a1c_max"] > 8),
}

overall_rate = y.mean() * 100
fig, ax = plt.subplots(figsize=(10, 8))
names = list(subgroups.keys())
rates = []
ns = []
for name, mask in subgroups.items():
    sub = df_feat_train[mask]
    rate = sub[outcome].mean() * 100 if len(sub) > 0 else 0
    rates.append(rate)
    ns.append(len(sub))

bars = ax.barh(range(len(names)), rates, color=C_UNCT, alpha=0.85, edgecolor="white")
for i, (bar, rate, n) in enumerate(zip(bars, rates, ns)):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f"{rate:.1f}% (n={n:,})", va="center", fontsize=9)
ax.axvline(overall_rate, color="gray", ls="--", lw=1.5, label=f"Overall ({overall_rate:.1f}%)")
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel("% Uncontrolled")
ax.set_title("Red Flag Subgroups: Uncontrolled Rate", fontweight="bold")
ax.legend(); ax.grid(alpha=0.2, axis="x")
ax.invert_yaxis()
fig.tight_layout(); plt.show()

# Print summary table
print(f"\n{'Subgroup':<50} {'n':>6} {'% Unctrl':>10} {'Mean A1C':>10}")
print("-" * 80)
for name, mask in subgroups.items():
    clean_name = name.replace("\n", " ")
    sub = df_feat_train[mask]
    n = len(sub)
    rate = sub[outcome].mean() * 100 if n > 0 else 0
    mean_a1c = sub["a1c_mean"].mean() if n > 0 else 0
    print(f"{clean_name:<50} {n:>6,} {rate:>9.1f}% {mean_a1c:>10.1f}")

# ── CELL 6E: Post-cleaning - 2025 measurement recency ──
ref_days = df_feat_train["a1c 2025-collection date-time-days from reference"]
last_lookback = df_feat_train[A1C_DAY_COLS_LOOKBACK].max(axis=1)
gap = ref_days - last_lookback

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: 2025 timing distribution
ax = axes[0, 0]
ax.hist(ref_days[y==0].dropna(), bins=50, alpha=0.5, color=C_CTRL, density=True, label="Controlled")
ax.hist(ref_days[y==1].dropna(), bins=50, alpha=0.5, color=C_UNCT, density=True, label="Uncontrolled")
ax.set_xlabel("Days from Index to 2025 A1C"); ax.set_ylabel("Density")
ax.set_title("2025 A1C Timing Distribution", fontweight="bold"); ax.legend(); ax.grid(alpha=0.2)

# Panel 2: Days to 2025 box by outcome
ax = axes[0, 1]
c_days = ref_days[y==0].dropna(); u_days = ref_days[y==1].dropna()
bp = ax.boxplot([c_days, u_days], labels=["Controlled", "Uncontrolled"], patch_artist=True, widths=0.5)
for patch, c in zip(bp["boxes"], [C_CTRL, C_UNCT]):
    patch.set_facecolor(c); patch.set_alpha(0.5)
ax.text(1, c_days.median(), f"med={c_days.median():.1f}", fontsize=8, ha="center",
        bbox=dict(boxstyle="round", fc=C_CTRL, alpha=0.3))
ax.text(2, u_days.median(), f"med={u_days.median():.1f}", fontsize=8, ha="center",
        bbox=dict(boxstyle="round", fc=C_UNCT, alpha=0.3))
ax.set_ylabel("Days to 2025 A1C"); ax.set_title("Days to 2025 A1C by Outcome", fontweight="bold")

# Panel 3: Gap distribution
ax = axes[1, 0]
c_gap = gap[y==0].dropna(); u_gap = gap[y==1].dropna()
ax.hist(c_gap, bins=50, alpha=0.5, color=C_CTRL, density=True,
        label=f"Controlled (med={c_gap.median():.0f}d)")
ax.hist(u_gap, bins=50, alpha=0.5, color=C_UNCT, density=True,
        label=f"Uncontrolled (med={u_gap.median():.0f}d)")
ax.set_xlabel("Days Between Last Lookback A1C and 2025 A1C"); ax.set_ylabel("Density")
ax.set_title("Gap: Last Lookback A1C to 2025 A1C", fontweight="bold"); ax.legend(fontsize=8); ax.grid(alpha=0.2)

# Panel 4: Outcome rate by 2025 timing bin
ax = axes[1, 1]
timing_bins = [0, 180, 270, 365, 450, 550, 700]
timing_labels = ["<180", "180-270", "270-365", "365-450", "450-550", "550+"]
df_feat_train["timing_grp"] = pd.cut(ref_days, bins=timing_bins, labels=timing_labels, right=True)
timing_rates = df_feat_train.groupby("timing_grp", observed=False)[outcome].mean() * 100
timing_ns = df_feat_train.groupby("timing_grp", observed=False)[outcome].count()
x_labels = [f"{l}\n(n={n:,})" for l, n in zip(timing_labels, timing_ns)]
bars = ax.bar(range(len(timing_rates)), timing_rates, color=C_UNCT, alpha=0.85, edgecolor="white")
for bar, rate in zip(bars, timing_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{rate:.1f}%", ha="center", fontsize=9)
ax.axhline(y.mean()*100, color="gray", ls="--", lw=1, alpha=0.6)
ax.set_xticks(range(len(timing_rates))); ax.set_xticklabels(x_labels, fontsize=8)
ax.set_ylabel("% Uncontrolled"); ax.set_title("Outcome Rate by 2025 A1C Timing", fontweight="bold")
ax.grid(alpha=0.2, axis="y")

fig.suptitle("Recency of 2025 A1C Measurement", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout(); plt.show()

# Clean up temp columns
df_feat_train.drop(columns=["age_group_post", "timing_grp"], inplace=True, errors="ignore")

# ── CELL 6F: Post-cleaning correlation matrix + feature importance + PDPs ──
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.inspection import PartialDependenceDisplay

corr_cols_post = ["age", "a1c 1-estimated result", "a1c 2-estimated result",
    "a1c_mean", "a1c_max", "a1c_range", "a1c_std", "a1c_delta_1_2", "n_a1c_visits",
    "bmi", "ldl-estimated result", "hdl-estimated result", "total cholesterol-estimated result",
    "cad-count", "copd-count", "ed vist count-count", "pcp visit count-count",
    "admission count-count", "insulin orders-count", "metformin orders-count",
    "glp-1 orders-count", "sglt2 orders-count", "sulfonylurea orders-count",
    "adi_national_rank"]
corr_labels_post = ["Age", "A1C-1", "A1C-2", "A1C Mean", "A1C Max", "A1C Range",
    "A1C Std", "Delta 1-2", "# Visits", "BMI", "LDL", "HDL", "TotChol",
    "CAD", "COPD", "ED", "PCP", "Admits", "Insulin", "Metformin",
    "GLP-1", "SGLT2", "Sulf", "ADI-Nat"]

valid_post = [c for c in corr_cols_post if c in df_feat_train.columns]
valid_labels_post = [corr_labels_post[i] for i, c in enumerate(corr_cols_post) if c in df_feat_train.columns]
corr_df_post = df_feat_train[valid_post + [outcome]].copy()
corr_df_post[outcome] = corr_df_post[outcome].astype(int)
corr_df_post.columns = valid_labels_post + ["Uncontrolled"]

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(corr_df_post.corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, ax=ax, annot_kws={"size": 5}, linewidths=0.5, vmin=-1, vmax=1)
ax.set_title("Correlation Matrix (Numeric Features + Outcome)", fontweight="bold", fontsize=14)
fig.tight_layout(); plt.show()

# Feature importance on cleaned data
print("\nTraining gradient boosting on cleaned data for feature importance and PDPs...")
from sklearn.impute import SimpleImputer
feat_cols_post = [c for c in valid_post]
X_post = df_feat_train[feat_cols_post].copy()
imp_post = SimpleImputer(strategy="median")
X_imp_post = pd.DataFrame(imp_post.fit_transform(X_post), columns=feat_cols_post, index=X_post.index)

gb_post = GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                                      subsample=0.8, random_state=42)
gb_post.fit(X_imp_post, y)

importances_post = gb_post.feature_importances_
sorted_idx_post = np.argsort(importances_post)
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh([valid_labels_post[i] for i in sorted_idx_post],
        importances_post[sorted_idx_post], color="#3b82f6", alpha=0.85)
ax.set_xlabel("Feature Importance (Gini)")
ax.set_title("Gradient Boosting Feature Importance", fontweight="bold")
ax.grid(alpha=0.2, axis="x")
fig.tight_layout(); plt.show()

# PDPs - top 6
top_6_post = np.argsort(-importances_post)[:6]
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, feat_i in zip(axes.flatten(), top_6_post):
    PartialDependenceDisplay.from_estimator(gb_post, X_imp_post, [feat_i], ax=ax,
        feature_names=valid_labels_post, grid_resolution=30)
    ax.set_title(valid_labels_post[feat_i], fontweight="bold")
fig.suptitle("Partial Dependence Plots (Top 6 Features)", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout(); plt.show()

# PDPs - remaining
remaining_post = np.argsort(-importances_post)[6:]
n_rem = len(remaining_post); n_cols_r = 3; n_rows_r = int(np.ceil(n_rem / n_cols_r))
fig, axes = plt.subplots(n_rows_r, n_cols_r, figsize=(16, 4 * n_rows_r))
axes_flat = axes.flatten()
for ax, feat_i in zip(axes_flat[:n_rem], remaining_post):
    PartialDependenceDisplay.from_estimator(gb_post, X_imp_post, [feat_i], ax=ax,
        feature_names=valid_labels_post, grid_resolution=30)
    ax.set_title(valid_labels_post[feat_i], fontweight="bold")
for ax in axes_flat[n_rem:]:
    ax.set_visible(False)
fig.suptitle("Partial Dependence Plots (Remaining Features)", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout(); plt.show()

print("Post-cleaning EDA complete.")

## Bug Fix Documentation: `visits_per_month` Inflation

The original computation divided visit count by observation span with no minimum span requirement. Patients with multiple A1c measurements taken days apart (e.g., 2 visits over 5 days) produced wildly inflated values like 12-30+ visits/month. The fix requires at least 3 months of observation span, setting the feature to NaN otherwise.

In [ ]:
# ── CELL 6B: visits_per_month bug demonstration ──
print("="*70)
print("BUG FIX: visits_per_month INFLATION")
print("="*70)

# Recompute using the ORIGINAL (buggy) formula: no minimum span
a1c_vals_bug = df_feat_train[A1C_RESULT_COLS]
n_visits_bug = a1c_vals_bug.notna().sum(axis=1).clip(lower=1)
day1_bug = df_feat_train["a1c 1-collection date-time-days from reference"]
last_day_bug = df_feat_train[A1C_DAY_COLS_LOOKBACK].max(axis=1)
obs_span_buggy = (last_day_bug - day1_bug) / 30.44

# Buggy version: no guard, any span > 0 gets a value
vpm_buggy = np.where(
    obs_span_buggy > 0,
    n_visits_bug / obs_span_buggy,
    np.nan
)
vpm_buggy = pd.Series(vpm_buggy, index=df_feat_train.index)

# Fixed version (already in df_feat_train)
vpm_fixed = df_feat_train["visits_per_month"]

print(f"\nBUGGY VERSION (no minimum span):")
print(f"  Valid values:  {vpm_buggy.notna().sum():,}")
print(f"  Mean:          {vpm_buggy.mean():.3f}")
print(f"  Median:        {vpm_buggy.median():.3f}")
print(f"  Max:           {vpm_buggy.max():.1f}")
print(f"  > 5/month:     {(vpm_buggy > 5).sum():,}")
print(f"  > 10/month:    {(vpm_buggy > 10).sum():,}")
print(f"  > 30/month:    {(vpm_buggy > 30).sum():,}")

print(f"\nFIXED VERSION (>= 3 months observation span required):")
print(f"  Valid values:  {vpm_fixed.notna().sum():,}")
print(f"  Mean:          {vpm_fixed.mean():.3f}")
print(f"  Median:        {vpm_fixed.median():.3f}")
print(f"  Max:           {vpm_fixed.max():.3f}")
print(f"  > 5/month:     {(vpm_fixed > 5).sum():,}")
print(f"  Set to NaN:    {vpm_fixed.isna().sum():,} (patients with < 3 months span)")

# Show worst offenders
print(f"\nTop 10 most inflated patients (buggy version):")
worst = vpm_buggy.nlargest(10)
for pid, val in worst.items():
    span_days = obs_span_buggy.loc[pid] * 30.44
    nv = int(n_visits_bug.loc[pid])
    fixed_val = vpm_fixed.loc[pid]
    fixed_str = f"{fixed_val:.3f}" if pd.notna(fixed_val) else "NaN (excluded)"
    print(f"  Patient {pid}: {val:.1f}/month (buggy) -> {fixed_str} (fixed)"
          f"  [{nv} visits over {span_days:.0f} days]")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

ax = axes[0]
ax.hist(vpm_buggy.dropna().clip(upper=15), bins=50, color="#ef4444", alpha=0.7, edgecolor="white")
ax.set_title("Buggy: no minimum span", fontweight="bold")
ax.set_xlabel("Visits per month")
ax.set_ylabel("Patients")
ax.axvline(5, color="black", ls="--", lw=1, label="> 5/month")
ax.legend(fontsize=9)

ax = axes[1]
ax.hist(vpm_fixed.dropna(), bins=50, color="#10b981", alpha=0.7, edgecolor="white")
ax.set_title("Fixed: >= 3 months required", fontweight="bold")
ax.set_xlabel("Visits per month")

ax = axes[2]
ctrl = vpm_fixed[y == 0].dropna()
unctrl = vpm_fixed[y == 1].dropna()
bp = ax.boxplot([ctrl, unctrl], labels=["Controlled", "Uncontrolled"],
                patch_artist=True, widths=0.5, showfliers=False)
for patch, c in zip(bp["boxes"], ["#10b981", "#ef4444"]):
    patch.set_facecolor(c); patch.set_alpha(0.6)
ax.set_ylabel("Visits per month")
ax.set_title("Fixed: by 2025 outcome", fontweight="bold")

fig.suptitle("visits_per_month Bug Fix", fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

print(f"\nThe buggy version inflated single-visit and short-span patients to")
print(f"impossible cadences. The fix sets visits_per_month to NaN for patients")
print(f"with < 3 months of observation, preventing the denominator from being")
print(f"near-zero.")

## Phase 5: Model Comparison (6 Models x 5-Fold Stratified CV)

Compare Linear Regression, Logistic Regression, Random Forest, LightGBM, XGBoost, and CatBoost on the minimal 25-feature set. All tree models use matched hyperparameters. Threshold: Youden's J per fold for binary predictions.

In [ ]:
# ── CELL 7: Model definitions ──
MODELS = [
    ("Linear Regression", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc",  StandardScaler()),
        ("lr",  LinearRegression()),
    ]), True),

    ("Logistic Regression", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc",  StandardScaler()),
        ("lr",  LogisticRegression(max_iter=1000, random_state=42)),
    ]), False),

    ("Random Forest", RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=20,
        random_state=42, n_jobs=-1,
    ), False),

    ("LightGBM", lgb.LGBMClassifier(
        n_estimators=500, max_depth=5, learning_rate=0.03,
        subsample=0.8, min_child_samples=20, num_leaves=40,
        reg_alpha=0.05, reg_lambda=0.5,
        random_state=42, verbose=-1, n_jobs=-1,
    ), False),

    ("XGBoost", xgb.XGBClassifier(
        n_estimators=500, max_depth=5, learning_rate=0.03,
        subsample=0.8, reg_alpha=0.05, reg_lambda=0.5,
        random_state=42, verbosity=0, n_jobs=-1,
        enable_categorical=False, eval_metric="logloss",
    ), False),

    ("CatBoost", cb.CatBoostClassifier(
        iterations=500, depth=5, learning_rate=0.03,
        subsample=0.8, l2_leaf_reg=1.0,
        random_state=42, verbose=0, thread_count=-1,
    ), False),
]

print(f"Models configured: {[m[0] for m in MODELS]}")

In [ ]:
# ── CELL 8: Run 5-fold CV for all models ──
print("=" * 75)
print("5-FOLD STRATIFIED CV: 6 MODELS x 25 FEATURES")
print("=" * 75)

all_results = {}

for model_name, model_obj, is_linreg in MODELS:
    print(f"\n{'─'*60}")
    print(f"  {model_name}")
    print(f"{'─'*60}")

    fold_data = {
        "auc": [], "pr_auc": [], "brier": [],
        "sens": [], "spec": [], "prec": [], "npv": [], "f1": [], "acc": [],
        "tp": [], "fp": [], "tn": [], "fn": [],
        "y_val_all": [], "proba_all": [],
    }

    for fold_i, (train_idx, val_idx) in enumerate(skf.split(X, strat_key)):
        X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]

        mod = clone(model_obj)
        if model_name == "LightGBM":
            mod.fit(X_tr, y_tr, categorical_feature=cat_idx)
        elif model_name == "CatBoost":
            mod.fit(X_tr, y_tr, cat_features=cat_idx)
        else:
            mod.fit(X_tr, y_tr)

        if is_linreg:
            proba = np.clip(mod.predict(X_va), 0, 1)
        else:
            proba = mod.predict_proba(X_va)[:, 1]

        auc = roc_auc_score(y_va, proba)
        pr_auc = average_precision_score(y_va, proba)
        brier = brier_score_loss(y_va, proba)

        fpr, tpr, thresh = roc_curve(y_va, proba)
        opt_t = thresh[np.argmax(tpr - fpr)]
        pred = (proba >= opt_t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_va, pred).ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        prec_val = tp / (tp + fp) if (tp + fp) > 0 else 0
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0
        f1 = 2 * prec_val * sens / (prec_val + sens) if (prec_val + sens) > 0 else 0

        fold_data["auc"].append(auc)
        fold_data["pr_auc"].append(pr_auc)
        fold_data["brier"].append(brier)
        fold_data["sens"].append(sens)
        fold_data["spec"].append(spec)
        fold_data["prec"].append(prec_val)
        fold_data["npv"].append(npv)
        fold_data["f1"].append(f1)
        fold_data["acc"].append(accuracy_score(y_va, pred))
        fold_data["tp"].append(tp); fold_data["fp"].append(fp)
        fold_data["tn"].append(tn); fold_data["fn"].append(fn)
        fold_data["y_val_all"].append(y_va.values)
        fold_data["proba_all"].append(proba)

        print(f"  Fold {fold_i+1}: AUC={auc:.4f}  PR={pr_auc:.4f}  Brier={brier:.4f}")

    all_results[model_name] = fold_data
    print(f"  Mean AUC: {np.mean(fold_data['auc']):.4f} "
          f"(+/-{np.std(fold_data['auc']):.4f})")

In [ ]:
# ── CELL 9: Summary table ──
print("\n" + "=" * 110)
print("MODEL COMPARISON SUMMARY (5-Fold StratifiedKFold CV, mean +/- std)")
print("=" * 110)

header = (f"{'Model':<22} {'AUC':>14} {'PR AUC':>14} {'Brier':>10} "
          f"{'Sens':>8} {'Spec':>8} {'Prec':>8} {'NPV':>8}")
print(header)
print("-" * 110)

sorted_models = sorted(all_results.keys(),
                       key=lambda m: -np.mean(all_results[m]["auc"]))

for name in sorted_models:
    m = all_results[name]
    def fmt(key):
        return f"{np.mean(m[key]):.4f}({np.std(m[key]):.3f})"
    def fmt3(key):
        return f"{np.mean(m[key]):.3f}"
    print(f"{name:<22} {fmt('auc'):>14} {fmt('pr_auc'):>14} {fmt('brier'):>10} "
          f"{fmt3('sens'):>8} {fmt3('spec'):>8} {fmt3('prec'):>8} {fmt3('npv'):>8}")

print("-" * 110)
print(f"Best model by AUC: {sorted_models[0]}")

## Phase 6: Discrimination Curves and Calibration

In [ ]:
# ── CELL 10: ROC and PR curves ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
palette = ["#8b5cf6", "#10b981", "#f97316", "#ef4444", "#3b82f6", "#6b7280"]

ax = axes[0]
for (name, color) in zip(sorted_models, palette):
    m = all_results[name]
    y_true = np.concatenate(m["y_val_all"])
    y_prob = np.concatenate(m["proba_all"])
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val = roc_auc_score(y_true, y_prob)
    lw = 2.5 if name == "CatBoost" else 1.5
    ax.plot(fpr, tpr, color=color, lw=lw, label=f"{name} ({auc_val:.4f})")
ax.plot([0,1],[0,1],"k--",lw=0.7,alpha=0.5)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves (pooled 5-fold)", fontweight="bold")
ax.legend(fontsize=9, loc="lower right"); ax.grid(alpha=0.2)

ax = axes[1]
base_rate = y.mean()
for (name, color) in zip(sorted_models, palette):
    m = all_results[name]
    y_true = np.concatenate(m["y_val_all"])
    y_prob = np.concatenate(m["proba_all"])
    prec_c, rec_c, _ = precision_recall_curve(y_true, y_prob)
    pr_auc = average_precision_score(y_true, y_prob)
    lw = 2.5 if name == "CatBoost" else 1.5
    ax.plot(rec_c, prec_c, color=color, lw=lw, label=f"{name} ({pr_auc:.4f})")
ax.axhline(base_rate, color="gray", ls="--", lw=0.8, alpha=0.5, label=f"Base rate ({base_rate:.3f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves (pooled 5-fold)", fontweight="bold")
ax.legend(fontsize=9, loc="upper right"); ax.grid(alpha=0.2)

fig.tight_layout()
plt.show()

## Phase 7: Permutation Importance Validation

Confirm that all 25 retained features contribute above the 0.0005 AUC-drop noise floor in the reduced feature space. Features below this threshold were already eliminated in the v7-to-v8 reduction (83 to 27 features, then refined to 25).

In [ ]:
# ── CELL 11: Permutation importance ──
print("="*60)
print("PERMUTATION IMPORTANCE (25-feature set)")
print("="*60)

# Fit production CatBoost on full training data
prod_model = cb.CatBoostClassifier(
    iterations=500, depth=5, learning_rate=0.03,
    subsample=0.8, l2_leaf_reg=1.0,
    random_state=42, verbose=0, thread_count=-1,
)
prod_model.fit(X, y, cat_features=cat_idx)

# Permutation importance on 5k sample
sample_idx = np.random.RandomState(42).choice(len(X), 5000, replace=False)
X_perm = X.iloc[sample_idx]
y_perm = y.iloc[sample_idx]

perm_result = permutation_importance(
    prod_model, X_perm, y_perm,
    scoring="roc_auc", n_repeats=10, random_state=42, n_jobs=-1,
)
perm_df = pd.DataFrame({
    "feature": MINIMAL_FEATURES,
    "auc_drop": perm_result.importances_mean,
    "std": perm_result.importances_std,
}).sort_values("auc_drop", ascending=False).reset_index(drop=True)

print(f"\n{'Feature':<40} {'AUC Drop':>12} {'Std':>10}")
print("-" * 65)
for _, row in perm_df.iterrows():
    marker = "* " if row["auc_drop"] < 0.0005 else "  "
    print(f"{marker}{row['feature']:<38} {row['auc_drop']:>12.5f} {row['std']:>10.5f}")

n_below = (perm_df["auc_drop"] < 0.0005).sum()
print(f"\nFeatures below 0.0005 threshold: {n_below}")

# Plot
fig, ax = plt.subplots(figsize=(9, 8))
top = perm_df.iloc[::-1]
colors = ["#ef4444" if v < 0.0005 else "#3b82f6" for v in top["auc_drop"]]
ax.barh(top["feature"], top["auc_drop"], xerr=top["std"],
        color=colors, capsize=3, alpha=0.85)
ax.axvline(0.0005, color="red", ls="--", lw=1, label="Noise floor (0.0005)")
ax.set_xlabel("AUC Drop When Shuffled")
ax.set_title("Permutation Importance (25 features)", fontweight="bold")
ax.legend(); ax.grid(alpha=0.2, axis="x")
fig.tight_layout()
plt.show()

## Quality Check: 5-Fold CV After Feature Reduction

Quick CatBoost-only CV to confirm the 25-feature reduced set maintains AUC parity with the earlier 83-feature v7 result (0.8758).

In [ ]:
# ── CELL 11B: Post-reduction quality check ──
print("="*70)
print("QUALITY CHECK: CatBoost 5-Fold CV on 25-Feature Set")
print("="*70)

qc_model = cb.CatBoostClassifier(
    iterations=500, depth=5, learning_rate=0.03,
    subsample=0.8, l2_leaf_reg=1.0,
    random_state=42, verbose=0, thread_count=-1,
)

qc_fold_aucs = []
for i, (tr_idx, va_idx) in enumerate(skf.split(X, strat_key)):
    mod = clone(qc_model)
    mod.fit(X.iloc[tr_idx], y.iloc[tr_idx], cat_features=cat_idx)
    proba = mod.predict_proba(X.iloc[va_idx])[:, 1]
    auc = roc_auc_score(y.iloc[va_idx], proba)
    qc_fold_aucs.append(auc)
    print(f"  Fold {i+1}: AUC = {auc:.4f}")

print(f"\n25-feature CV:  {np.mean(qc_fold_aucs):.4f} (+/-{np.std(qc_fold_aucs):.4f})")
print(f"83-feature v7:  0.8758 (+/-0.004)")
print(f"Delta:          {np.mean(qc_fold_aucs) - 0.8758:+.4f}")
print(f"\nWithin noise floor: {'Yes' if abs(np.mean(qc_fold_aucs) - 0.8758) < np.std(qc_fold_aucs) else 'No'}")

## Phase 8: Threshold Analysis and Production Model

Compute Youden's J threshold from out-of-fold predictions across all 5 folds. The threshold balances sensitivity and specificity, with priority on sensitivity (false negatives carry higher clinical cost in care management).

In [ ]:
# ── CELL 12: Threshold analysis on CatBoost OOF predictions ──
print("="*60)
print("THRESHOLD ANALYSIS (CatBoost OOF Predictions)")
print("="*60)

# Collect OOF predictions from CatBoost
oof_proba = np.zeros(len(y))
for tr_idx, va_idx in skf.split(X, strat_key):
    mod = clone(prod_model)
    mod.fit(X.iloc[tr_idx], y.iloc[tr_idx], cat_features=cat_idx)
    oof_proba[va_idx] = mod.predict_proba(X.iloc[va_idx])[:, 1]

# Youden's J
fpr_oof, tpr_oof, thresh_oof = roc_curve(y, oof_proba)
j_scores = tpr_oof - fpr_oof
best_j_idx = np.argmax(j_scores)
FINAL_THRESHOLD = thresh_oof[best_j_idx]

oof_pred = (oof_proba >= FINAL_THRESHOLD).astype(int)
tn, fp, fn, tp = confusion_matrix(y, oof_pred).ravel()

print(f"\nYouden's J threshold: {FINAL_THRESHOLD:.4f}")
print(f"  Sensitivity: {tp/(tp+fn):.3f}")
print(f"  Specificity: {tn/(tn+fp):.3f}")
print(f"  Precision:   {tp/(tp+fp):.3f}")
print(f"  NPV:         {tn/(tn+fn):.3f}")
print(f"  TP/FP/TN/FN: {tp} / {fp} / {tn} / {fn}")

# Calibration
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
prob_true, prob_pred = calibration_curve(y, oof_proba, n_bins=10, strategy="uniform")
ax.plot(prob_pred, prob_true, marker="o", lw=2, color="#8b5cf6", label="CatBoost")
ax.plot([0,1],[0,1],"k--",lw=0.5)
cal_mae = np.mean(np.abs(prob_true - prob_pred))
ax.set_title(f"Calibration (MAE={cal_mae:.4f})", fontweight="bold")
ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Fraction of positives")
ax.legend(); ax.grid(alpha=0.2)

ax = axes[1]
ax.hist(oof_proba[y==0], bins=50, alpha=0.6, color="#10b981", label="Controlled", density=True)
ax.hist(oof_proba[y==1], bins=50, alpha=0.6, color="#ef4444", label="Uncontrolled", density=True)
ax.axvline(FINAL_THRESHOLD, color="black", ls="--", lw=1.5, label=f"Threshold ({FINAL_THRESHOLD:.4f})")
ax.set_title("OOF Probability Distribution", fontweight="bold")
ax.set_xlabel("Predicted probability"); ax.set_ylabel("Density")
ax.legend(fontsize=9); ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

print(f"\nFINAL_THRESHOLD = {FINAL_THRESHOLD:.4f}")

## Phase 9: The Labeling Discovery and 5-Fold AUC Comparison

26,617 of 62,425 training patients (42.7%) have null 2025 collection dates, and every single one is labeled "controlled" (False). This is a deterministic labeling rule, not a clinical determination.

This section compares three 5-fold CV approaches:
1. Train on filtered cohort, predict on the full 62,425 (honest probabilities for everyone)
2. Train on filtered cohort, predict on the full 62,425 with null-date patients hardcoded to False / probability 0.0
3. Train and evaluate on the full 62,425 directly (model learns the labeling rule)

In [ ]:
# ── CELL 14: 5-Fold AUC comparison (3 approaches) ──
print("="*70)
print("5-FOLD AUC COMPARISON: HONEST vs HARDCODED vs FULL-COHORT")
print("="*70)

# Prepare the full 62,425 dataset for prediction
print("\nPreparing full 62,425 dataset for scoring...")
df_clean_all, _ = clean_features(df_full, drop_rows=False, verbose=False)
df_feat_all, _ = engineer_features(df_clean_all, training_race_vocab=training_race_vocab, verbose=False)
X_all = df_feat_all[MINIMAL_FEATURES].copy()
y_all = df_full[outcome].astype(int)
null_2025 = df_full[REF_COL].isna()
print(f"  Full dataset: {len(y_all):,} patients")
print(f"  Null 2025 date: {null_2025.sum():,} ({null_2025.mean()*100:.1f}%)")
print(f"  All null-date labeled controlled: {(y_all[null_2025.values] == 0).all()}")

model_template = cb.CatBoostClassifier(
    iterations=500, depth=5, learning_rate=0.03,
    subsample=0.8, l2_leaf_reg=1.0,
    random_state=42, verbose=0, thread_count=-1,
)

# ══════════════════════════════════════════════════════════
# APPROACH 1: Train on filtered cohort, score on ALL 62,425
#              (honest predictions for everyone)
# ══════════════════════════════════════════════════════════
print(f"\n{'='*70}")
print("APPROACH 1: Train on filtered, predict honestly on all 62,425")
print(f"{'='*70}")

fold_aucs_honest = []
for i, (tr_idx, va_idx) in enumerate(skf.split(X, strat_key)):
    mod = clone(model_template)
    mod.fit(X.iloc[tr_idx], y.iloc[tr_idx], cat_features=cat_idx)
    proba_all_fold = mod.predict_proba(X_all)[:, 1]
    auc = roc_auc_score(y_all, proba_all_fold)
    fold_aucs_honest.append(auc)
    print(f"  Fold {i+1}: AUC = {auc:.4f}")

print(f"  Mean: {np.mean(fold_aucs_honest):.4f} (+/-{np.std(fold_aucs_honest):.4f})")

# ══════════════════════════════════════════════════════════
# APPROACH 2: Train on filtered cohort, score on ALL 62,425
#              but hardcode null-date patients to 0.0
# ══════════════════════════════════════════════════════════
print(f"\n{'='*70}")
print("APPROACH 2: Train on filtered, hardcode null-date to False (prob=0.0)")
print(f"{'='*70}")

fold_aucs_hardcoded = []
for i, (tr_idx, va_idx) in enumerate(skf.split(X, strat_key)):
    mod = clone(model_template)
    mod.fit(X.iloc[tr_idx], y.iloc[tr_idx], cat_features=cat_idx)
    proba_all_fold = mod.predict_proba(X_all)[:, 1]
    proba_all_fold[null_2025.values] = 0.0
    auc = roc_auc_score(y_all, proba_all_fold)
    fold_aucs_hardcoded.append(auc)
    print(f"  Fold {i+1}: AUC = {auc:.4f}")

print(f"  Mean: {np.mean(fold_aucs_hardcoded):.4f} (+/-{np.std(fold_aucs_hardcoded):.4f})")

# ══════════════════════════════════════════════════════════
# APPROACH 3: Train and evaluate on ALL 62,425 directly
# ══════════════════════════════════════════════════════════
print(f"\n{'='*70}")
print("APPROACH 3: Train and evaluate on full 62,425 (learns labeling rule)")
print(f"{'='*70}")

df_feat_all["_visit_group"] = (df_feat_all["n_a1c_visits"] >= 2).astype(int)
strat_all = y_all.astype(str) + "_" + df_feat_all["_visit_group"].astype(str)
skf_all = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_aucs_full = []
for i, (tr_idx, va_idx) in enumerate(skf_all.split(X_all, strat_all)):
    mod = clone(model_template)
    mod.fit(X_all.iloc[tr_idx], y_all.iloc[tr_idx], cat_features=cat_idx)
    proba_v = mod.predict_proba(X_all.iloc[va_idx])[:, 1]
    auc = roc_auc_score(y_all.iloc[va_idx], proba_v)
    fold_aucs_full.append(auc)
    print(f"  Fold {i+1}: AUC = {auc:.4f}")

print(f"  Mean: {np.mean(fold_aucs_full):.4f} (+/-{np.std(fold_aucs_full):.4f})")

# ══════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════
print(f"\n{'='*70}")
print("SUMMARY")
print(f"{'='*70}")
print(f"{'Approach':<60} {'AUC':>10}")
print(f"{'-'*72}")
print(f"{'1. Filtered train, honest predict on all 62k':<60} {np.mean(fold_aucs_honest):>10.4f}")
print(f"{'2. Filtered train, null-date hardcoded to 0.0':<60} {np.mean(fold_aucs_hardcoded):>10.4f}")
print(f"{'3. Train and evaluate on full 62k':<60} {np.mean(fold_aucs_full):>10.4f}")
print(f"{'-'*72}")
print(f"{'Filtered cohort CV (quality check, reference)':<60} {np.mean(qc_fold_aucs):>10.4f}")

# Train production model on full filtered cohort for downstream use
prod_model_final = clone(model_template)
prod_model_final.fit(X, y, cat_features=cat_idx)
proba_honest = prod_model_final.predict_proba(X_all)[:, 1]
print(f"\nProduction model trained on {len(y):,} patients for submission pipeline.")

In [ ]:
# ── CELL 15: Null-date patient risk profile ──
print("="*70)
print("NULL-DATE PATIENT RISK PROFILE")
print("="*70)

null_proba = proba_honest[null_2025.values]
print(f"Total null-date patients: {len(null_proba):,}")
print(f"Mean predicted probability: {null_proba.mean():.4f}")
print(f"Median: {np.median(null_proba):.4f}")
print(f"\nPredicted uncontrolled (>{FINAL_THRESHOLD:.4f}): "
      f"{(null_proba > FINAL_THRESHOLD).sum():,} ({(null_proba > FINAL_THRESHOLD).mean()*100:.1f}%)")
print(f"High risk (>0.50): {(null_proba > 0.50).sum():,} ({(null_proba > 0.50).mean()*100:.1f}%)")
print(f"Very high risk (>0.80): {(null_proba > 0.80).sum():,} ({(null_proba > 0.80).mean()*100:.1f}%)")

print(f"\nProbability percentiles:")
for q in [10, 25, 50, 75, 90, 95, 99]:
    print(f"  P{q}: {np.percentile(null_proba, q):.4f}")

# Patient 6313 case study
print(f"\n{'='*70}")
print("PATIENT 6313 CASE STUDY")
print(f"{'='*70}")
if 6313 in df_feat_all.index:
    idx_6313 = list(df_feat_all.index).index(6313)
    p6313 = proba_honest[idx_6313]
    row = df_full.loc[6313]
    a1c_vals_6313 = [row[c] for c in A1C_RESULT_COLS if pd.notna(row[c])]
    print(f"  Age: {2025 - row['date of birth']:.0f}")
    print(f"  Race: {row['race - primary']}")
    print(f"  A1c values: {a1c_vals_6313}")
    print(f"  Insulin orders: {row['insulin orders-count']:.0f}")
    print(f"  BMI: {df_feat_all.loc[6313, 'bmi']:.1f}" if pd.notna(df_feat_all.loc[6313, 'bmi']) else "  BMI: N/A")
    print(f"  ADI national: {df_feat_all.loc[6313, 'adi_national_rank']:.0f}" if pd.notna(df_feat_all.loc[6313, 'adi_national_rank']) else "  ADI: N/A")
    print(f"  Null 2025 date: {null_2025.loc[6313]}")
    print(f"  Dataset label: {row[outcome]}")
    print(f"  Model probability: {p6313:.4f}")
    print(f"  Hardcoded would assign: 0.0000")

In [ ]:
# ── CELL 16: Visualization ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: probability distributions by 2025 date status
ax = axes[0]
prod_viz = cb.CatBoostClassifier(
    iterations=500, depth=5, learning_rate=0.03,
    subsample=0.8, l2_leaf_reg=1.0,
    random_state=42, verbose=0, thread_count=-1,
)
prod_viz.fit(X, y, cat_features=cat_idx)
proba_viz = prod_viz.predict_proba(X_all)[:, 1]

ax.hist(proba_viz[~null_2025.values], bins=50, alpha=0.7,
        color="#10b981", label="Has 2025 date", density=True)
ax.hist(proba_viz[null_2025.values], bins=50, alpha=0.7,
        color="#ef4444", label="Null 2025 date", density=True)
ax.axvline(FINAL_THRESHOLD, color="black", ls="--", lw=1.5,
           label=f"Threshold ({FINAL_THRESHOLD:.4f})")
ax.set_xlabel("Predicted probability")
ax.set_ylabel("Density")
ax.set_title("Honest predictions by 2025 date status", fontweight="bold")
ax.legend(fontsize=9); ax.grid(alpha=0.2)

# Right: AUC comparison bar chart
ax = axes[1]
approaches = ["Filtered train,\nhonest predict",
              "Filtered train,\nhardcoded null",
              "Train on\nfull 62k"]
aucs = [np.mean(fold_aucs_honest), np.mean(fold_aucs_hardcoded), np.mean(fold_aucs_full)]
stds = [np.std(fold_aucs_honest), np.std(fold_aucs_hardcoded), np.std(fold_aucs_full)]
colors = ["#10b981", "#ef4444", "#6b7280"]
bars = ax.bar(approaches, aucs, yerr=stds, color=colors, alpha=0.85,
              edgecolor="white", capsize=5)
for bar, auc_val in zip(bars, aucs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{auc_val:.4f}", ha="center", va="bottom", fontsize=11, fontweight=500)
ax.set_ylim(0.80, 1.02)
ax.set_ylabel("5-Fold CV AUC")
ax.set_title("AUC by prediction approach", fontweight="bold")
ax.grid(alpha=0.2, axis="y")

fig.tight_layout()
plt.show()

## Phase 10: Generate Final Submission (Honest Prediction)

Predict on the held-out test set (15,607 patients) using the production CatBoost model trained on the full filtered cohort. Every patient gets a real probability from the model. This is what we actually submitted.

In [ ]:
# ── CELL 17: Predict on test set ──
PREDICT_FILE = drive_path + "TEST_SET_DM_Features.csv"

print("="*70)
print(f"FINAL SUBMISSION: {PREDICT_FILE.split('/')[-1]}")
print("="*70)

test_raw = pd.read_csv(PREDICT_FILE, index_col=0)
original_ids = test_raw.index.values
print(f"Test set: {len(test_raw):,} patients")

# Clean (drop_rows=False: keep every row)
test_clean, _ = clean_features(test_raw, drop_rows=False, verbose=True)

# Engineer features using training vocabulary
test_feat, _ = engineer_features(test_clean, training_race_vocab=training_race_vocab, verbose=True)

# Predict
X_test = test_feat[MINIMAL_FEATURES].copy()
test_proba = prod_model_final.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= FINAL_THRESHOLD).astype(bool)

# Flags
flag_no_2025 = test_feat[REF_COL].isna()
flag_no_a1c = (test_feat[A1C_RESULT_COLS].notna().sum(axis=1) == 0)

print(f"\nPredictions:")
print(f"  Mean probability: {test_proba.mean():.4f}")
print(f"  Uncontrolled (True):  {test_pred.sum():,}")
print(f"  Controlled (False):   {(~test_pred).sum():,}")
print(f"  Null 2025 date:       {flag_no_2025.sum():,} ({flag_no_2025.mean()*100:.1f}%)")
print(f"  Threshold: {FINAL_THRESHOLD:.4f}")

# Write submission
submission = pd.DataFrame({
    "id": original_ids,
    "prediction": test_pred,
    "probability": test_proba,
})

audit = pd.DataFrame({
    "id": original_ids,
    "prediction": test_pred,
    "probability": test_proba,
    "flag_no_2025_date": flag_no_2025.values,
    "flag_no_valid_a1c": flag_no_a1c.values,
})

sub_path = drive_path + "Me_Myself_and_AI_Final_Submission.csv"
audit_path = drive_path + "v9_audit_qualitycheck.csv"
submission.to_csv(sub_path, index=False)
audit.to_csv(audit_path, index=False)

print(f"\nSaved: {sub_path}")
print(f"Saved: {audit_path}")
print(f"\nSubmission preview:")
print(submission.head(10).to_string(index=False))